# Hito 1 — Validación de carga en MongoDB Atlas
**1.1 Instalar dependencias**

instala las librerías necesarias para conectarse a MongoDB Atlas desde Google Colab y para presentar resultados en tablas.

In [ ]:
!pip install --upgrade "pymongo[srv]" dnspython certifi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.6 MB/s eta 0:00:00


**1.2 Crear conexión segura a MongoDB Atlas**

Explicación: esta celda valida que Colab puede conectarse realmente al cluster de Atlas. El ping confirma que las credenciales, el string de conexión y el acceso de red están funcionando.

In [ ]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from urllib.parse import quote_plus
from getpass import getpass
import certifi

MONGO_PASSWORD = quote_plus(getpass("Password MongoDB Atlas: "))

MONGO_CLUSTER = "ecommify-cluster.neewl8q.mongodb.net"

uri = (
    f"mongodb+srv://ecommify_user:{MONGO_PASSWORD}@{MONGO_CLUSTER}/"
    "?retryWrites=true&w=majority&appName=ecommify-cluster"
)

client = MongoClient(
    uri,
    tls=True,
    tlsCAFile=certifi.where(),
    server_api=ServerApi("1"),
    serverSelectionTimeoutMS=30000
)

client.admin.command("ping")

print("Conexión exitosa a MongoDB Atlas")

Password MongoDB Atlas: ··········
Conexión exitosa a MongoDB Atlas


**1.3 Seleccionar base de datos y colecciones esperadas**

Esta celda comprueba que las colecciones necesarias para la etapa existen en la base ecommify_mongodb. Si product_catalog y product_reviews aparecen, no sería necesario recargarlas.

In [ ]:
db = client["ecommify_mongodb"]

expected_collections = ["product_catalog", "product_reviews"]

collections_found = db.list_collection_names()

print("Colecciones encontradas:")
for collection in collections_found:
    print("-", collection)

missing = [c for c in expected_collections if c not in collections_found]

if missing:
    print("\nColecciones faltantes:", missing)
else:
    print("\nLas colecciones principales de Ecommify están disponibles.")

Colecciones encontradas:
- connection_test
- product_reviews
- product_catalog

Las colecciones principales de Ecommify están disponibles.


**1.4 Contar documentos reales por colección**

Atlas muestra conteos aproximados como “1K”, pero esta celda obtiene el número exacto de documentos. Esta tabla sirve como evidencia directa para el documento Word.

In [ ]:
import pandas as pd

summary = []

for collection_name in expected_collections:
    collection = db[collection_name]
    total_docs = collection.count_documents({})

    summary.append({
        "collection": collection_name,
        "total_documents": total_docs
    })

df_counts = pd.DataFrame(summary)
df_counts

,collection,total_documents
0,product_catalog,1000
1,product_reviews,102172


**1.5 Revisar estructura de un documento de muestra**

esta celda permite verificar que los documentos de product_catalog tienen la estructura esperada: datos básicos del producto, categoría, precio, especificaciones, vendedor, rating, ventas y campos de búsqueda.

In [ ]:
sample_doc = db["product_catalog"].find_one()

sample_doc

{'_id': '1e9e8ef04dbcff4541ed26657ea517e5',
 'product_id': '1e9e8ef04dbcff4541ed26657ea517e5',
 'name': 'Product perfumery 1e9e8ef0',
 'category': {'name': 'perfumaria', 'name_translated': 'perfumery'},
 'price_summary': {'min_price': 10.91,
  'max_price': 10.91,
  'avg_price': 10.91,
  'currency': 'BRL'},
 'specifications': {'product_name_length': 40,
  'product_description_length': 287,
  'photos_qty': 1,
  'package_profile': {'weight_g': 225.0,
   'length_cm': 16.0,
   'height_cm': 10.0,
   'width_cm': 14.0}},
 'physical_attributes': {'weight_g': 225.0,
  'length_cm': 16.0,
  'height_cm': 10.0,
  'width_cm': 14.0},
 'seller_summary': [{'seller_id': '5670f4db5b62c43d542e1b2d56b0cf7c',
   'city': 'sao paulo',
   'state': 'SP',
   'units_sold_by_seller': 1,
   'avg_freight_value': 7.39}],
 'rating_summary': {'average_rating': 5.0,
  'total_reviews': 1,
  'last_review_date': '2018-04-28T00:00:00'},
 'sales_summary': {'total_units_sold': 1,
  'total_orders': 1,
  'last_order_date': '2018

**1.6 Validar campos principales de product_catalog**

Esta celda revisa si el documento de muestra contiene los campos mínimos esperados para la optimización del catálogo. Esto ayuda a confirmar que la colección no solo existe, sino que tiene una estructura útil para consultas, índices y pipelines.

In [ ]:
required_fields = [
    "product_id",
    "name",
    "category",
    "price_summary",
    "specifications",
    "seller_summary",
    "rating_summary",
    "sales_summary",
    "search_keywords",
    "created_at",
    "updated_at"
]

sample_doc = db["product_catalog"].find_one()

field_validation = []

for field in required_fields:
    field_validation.append({
        "field": field,
        "exists_in_sample": field in sample_doc
    })

df_fields = pd.DataFrame(field_validation)
df_fields

,field,exists_in_sample
0,product_id,True
1,name,True
2,category,True
3,price_summary,True
4,specifications,True
5,seller_summary,True
6,rating_summary,True
7,sales_summary,True
8,search_keywords,True
9,created_at,True


**1.7 Revisar índices existentes**

Esta celda lista los índices existentes en product_catalog y product_reviews. Es importante porque antes de crear nuevos índices debemos saber qué ya existe, evitar duplicados y preparar la línea base de optimización.

In [ ]:
index_summary = []

for collection_name in expected_collections:
    collection = db[collection_name]
    indexes = list(collection.list_indexes())

    for index in indexes:
        index_summary.append({
            "collection": collection_name,
            "index_name": index.get("name"),
            "keys": index.get("key"),
            "unique": index.get("unique", False),
            "partial_filter": index.get("partialFilterExpression", None)
        })

df_indexes = pd.DataFrame(index_summary)
df_indexes

,collection,index_name,keys,unique,partial_filter
0,product_catalog,_id_,{'_id': 1},False,None
1,product_catalog,idx_category_name_translated,{'category.name_translated': 1},False,None
2,product_catalog,idx_total_units_sold,{'sales_summary.total_units_sold': -1},False,None
3,product_catalog,idx_average_rating,{'rating_summary.average_rating': -1},False,None
4,product_catalog,idx_seller_summary_seller_id,{'seller_summary.seller_id': 1},False,None
5,product_catalog,idx_category_rating,"{'category.name_translated': 1, 'rating_summar...",False,None
6,product_catalog,idx_text_product_search,"{'_fts': 'text', '_ftsx': 1}",False,None
7,product_catalog,idx_catalog_category_sales,"{'category.name_translated': 1, 'sales_summary...",False,None
8,product_catalog,idx_catalog_category_price,"{'category.name_translated': 1, 'price_summary...",False,None
9,product_catalog,idx_catalog_seller_name,"{'seller_summary.seller_id': 1, 'name': 1}",False,None


**1.8 Diagnóstico final del Hito 1**

Esta celda resume la validación del hito en un solo resultado. Es ideal para tomar captura y usarla después como evidencia en el documento.

In [ ]:
diagnostic = {
    "database": "ecommify_mongodb",
    "product_catalog_exists": "product_catalog" in collections_found,
    "product_reviews_exists": "product_reviews" in collections_found,
    "product_catalog_count": db["product_catalog"].count_documents({}),
    "product_reviews_count": db["product_reviews"].count_documents({}),
    "product_catalog_indexes": len(list(db["product_catalog"].list_indexes())),
    "product_reviews_indexes": len(list(db["product_reviews"].list_indexes()))
}

diagnostic

{'database': 'ecommify_mongodb',
 'product_catalog_exists': True,
 'product_reviews_exists': True,
 'product_catalog_count': 1000,
 'product_reviews_count': 102172,
 'product_catalog_indexes': 10,
 'product_reviews_indexes': 9}

# Hito 2 — Definición de consultas críticas MongoDB
**2.1 Seleccionar base de datos y colecciones**

Esta celda deja listas las dos colecciones principales que se van a optimizar en la etapa: product_catalog para catálogo enriquecido y product_reviews para reseñas. No se crean datos nuevos; solamente se reutilizan las colecciones existentes.

In [ ]:
# Hito 2.1 - Selección de base de datos y colecciones principales

db = client["ecommify_mongodb"]

product_catalog = db["product_catalog"]
product_reviews = db["product_reviews"]

print("Base de datos seleccionada:", db.name)
print("Colecciones de trabajo:")
print("-", product_catalog.name)
print("-", product_reviews.name)

Base de datos seleccionada: ecommify_mongodb
Colecciones de trabajo:
- product_catalog
- product_reviews


**2.2 Revisar la estructura de campos disponibles**

Esta celda muestra los campos reales disponibles dentro de los documentos. Es útil porque category, price_summary, rating_summary, sales_summary y seller_summary son objetos o arreglos, así que necesitamos conocer sus rutas internas antes de definir consultas críticas.


In [ ]:
# Hito 2.2 - Exploración de campos disponibles en documentos de muestra

import pandas as pd

def flatten_keys(document, parent_key="", separator="."):
    """
    Convierte un documento anidado en una lista de rutas tipo:
    category.name_translated, price_summary.avg_price, etc.
    """
    keys = []

    if isinstance(document, dict):
        for key, value in document.items():
            new_key = f"{parent_key}{separator}{key}" if parent_key else key
            keys.append(new_key)

            if isinstance(value, dict):
                keys.extend(flatten_keys(value, new_key, separator))
            elif isinstance(value, list) and len(value) > 0 and isinstance(value[0], dict):
                keys.extend(flatten_keys(value[0], new_key, separator))

    return keys


sample_catalog = product_catalog.find_one()
sample_reviews = product_reviews.find_one()

catalog_fields_df = pd.DataFrame({
    "collection": "product_catalog",
    "field_path": flatten_keys(sample_catalog)
})

reviews_fields_df = pd.DataFrame({
    "collection": "product_reviews",
    "field_path": flatten_keys(sample_reviews)
})

display(catalog_fields_df)
display(reviews_fields_df)

,collection,field_path
0,product_catalog,_id
1,product_catalog,product_id
2,product_catalog,name
3,product_catalog,category
4,product_catalog,category.name
5,product_catalog,category.name_translated
6,product_catalog,price_summary
7,product_catalog,price_summary.min_price
8,product_catalog,price_summary.max_price
9,product_catalog,price_summary.avg_price


,collection,field_path
0,product_reviews,_id
1,product_reviews,review_id
2,product_reviews,order_id
3,product_reviews,product_id
4,product_reviews,seller_id
5,product_reviews,customer_id
6,product_reviews,score
7,product_reviews,title
8,product_reviews,message
9,product_reviews,sentiment


**2.3 Definir campos candidatos para las consultas**

Esta celda selecciona automáticamente los campos que se usarán en las consultas críticas. Si algún nombre cambia frente a lo esperado, el notebook lo deja visible en la tabla para corregirlo antes de continuar.

In [ ]:
# Hito 2.3 - Definición controlada de campos usados por las consultas críticas

CATALOG_FIELDS = set(catalog_fields_df["field_path"].tolist())
REVIEWS_FIELDS = set(reviews_fields_df["field_path"].tolist())

def pick_field(available_fields, candidates):
    """
    Retorna el primer campo candidato que exista en la colección.
    Si ninguno existe, retorna None.
    """
    for field in candidates:
        if field in available_fields:
            return field
    return None


category_field = pick_field(CATALOG_FIELDS, [
    "category.name_translated",
    "category.name",
    "category"
])

price_field = pick_field(CATALOG_FIELDS, [
    "price_summary.avg_price",
    "price_summary.average_price",
    "price_summary.mean_price",
    "price_summary.min_price",
    "price_summary.max_price",
    "price_summary.price"
])

rating_field = pick_field(CATALOG_FIELDS, [
    "rating_summary.avg_score",
    "rating_summary.average_score",
    "rating_summary.review_score",
    "rating_summary.rating"
])

sales_field = pick_field(CATALOG_FIELDS, [
    "sales_summary.total_sales",
    "sales_summary.total_orders",
    "sales_summary.units_sold",
    "sales_summary.count"
])

seller_field = pick_field(CATALOG_FIELDS, [
    "seller_summary.seller_id",
    "seller_summary.id"
])

review_score_field = pick_field(REVIEWS_FIELDS, [
    "review_score",
    "score",
    "rating"
])

review_date_field = pick_field(REVIEWS_FIELDS, [
    "review_creation_date",
    "review_created_at",
    "created_at"
])

field_selection_df = pd.DataFrame([
    {"logical_field": "category", "mongo_field": category_field},
    {"logical_field": "price", "mongo_field": price_field},
    {"logical_field": "rating", "mongo_field": rating_field},
    {"logical_field": "sales", "mongo_field": sales_field},
    {"logical_field": "seller", "mongo_field": seller_field},
    {"logical_field": "review_score", "mongo_field": review_score_field},
    {"logical_field": "review_date", "mongo_field": review_date_field},
])

field_selection_df

,logical_field,mongo_field
0,category,category.name_translated
1,price,price_summary.avg_price
2,rating,None
3,sales,sales_summary.total_orders
4,seller,seller_summary.seller_id
5,review_score,score
6,review_date,created_at


**2.4 Obtener valores reales para las consultas**

esta celda toma valores reales desde MongoDB para evitar consultas inventadas. Así las consultas críticas se prueban contra datos existentes del catálogo y reseñas de Ecommify.

In [ ]:
# Hito 2.4 - Obtención de valores reales para parametrizar consultas

selected_category = None
selected_product_id = None
selected_seller_id = None

if category_field:
    categories = [c for c in product_catalog.distinct(category_field) if c is not None]
    selected_category = categories[0] if categories else None

sample_product = product_catalog.find_one({"product_id": {"$exists": True}})
if sample_product:
    selected_product_id = sample_product.get("product_id")

# Intento de extracción de seller_id desde seller_summary
sample_with_seller = product_catalog.find_one({"seller_summary": {"$exists": True, "$ne": []}})
if sample_with_seller:
    seller_summary = sample_with_seller.get("seller_summary", [])
    if isinstance(seller_summary, list) and len(seller_summary) > 0:
        selected_seller_id = seller_summary[0].get("seller_id") or seller_summary[0].get("id")

parameters_df = pd.DataFrame([
    {"parameter": "selected_category", "value": selected_category},
    {"parameter": "selected_product_id", "value": selected_product_id},
    {"parameter": "selected_seller_id", "value": selected_seller_id},
])

parameters_df

,parameter,value
0,selected_category,agro_industry_and_commerce
1,selected_product_id,1e9e8ef04dbcff4541ed26657ea517e5
2,selected_seller_id,5670f4db5b62c43d542e1b2d56b0cf7c


**2.5 Definir consultas críticas Q01-Q07**

Esta celda define siete consultas críticas del módulo MongoDB. Cubren navegación de catálogo, filtros por categoría/precio, consulta por vendedor, búsqueda textual, reseñas por producto, reseñas críticas y análisis por categoría. Estas consultas serán la base de medición y optimización de los siguientes hitos.

In [ ]:
# Hito 2.5 - Definición de consultas críticas para optimización

critical_queries = []

# Q01 - Navegación de catálogo por categoría
q01_filter = {}
if category_field and selected_category:
    q01_filter[category_field] = selected_category

q01_sort = [(sales_field, -1)] if sales_field else [("name", 1)]

critical_queries.append({
    "code": "Q01",
    "collection": "product_catalog",
    "operation": "find",
    "business_purpose": "Consultar productos por categoría, ordenados por relevancia comercial.",
    "filter": q01_filter,
    "sort": q01_sort,
    "projection": {
        "_id": 0,
        "product_id": 1,
        "name": 1,
        "category": 1,
        "price_summary": 1,
        "sales_summary": 1,
        "rating_summary": 1
    },
    "index_goal": "Índice compuesto para categoría y ordenamiento."
})

# Q02 - Filtro por categoría y rango de precio
q02_filter = {}
if category_field and selected_category:
    q02_filter[category_field] = selected_category
if price_field:
    q02_filter[price_field] = {"$gte": 50}

q02_sort = [(price_field, 1)] if price_field else [("name", 1)]

critical_queries.append({
    "code": "Q02",
    "collection": "product_catalog",
    "operation": "find",
    "business_purpose": "Filtrar productos por categoría y precio mínimo.",
    "filter": q02_filter,
    "sort": q02_sort,
    "projection": {
        "_id": 0,
        "product_id": 1,
        "name": 1,
        "category": 1,
        "price_summary": 1
    },
    "index_goal": "Índice compuesto aplicando regla ESR: igualdad, ordenamiento y rango."
})

# Q03 - Consulta por vendedor
q03_filter = {}
if selected_seller_id:
    q03_filter["seller_summary.seller_id"] = selected_seller_id
elif category_field and selected_category:
    q03_filter[category_field] = selected_category

critical_queries.append({
    "code": "Q03",
    "collection": "product_catalog",
    "operation": "find",
    "business_purpose": "Consultar productos asociados a un vendedor.",
    "filter": q03_filter,
    "sort": [("name", 1)],
    "projection": {
        "_id": 0,
        "product_id": 1,
        "name": 1,
        "seller_summary": 1,
        "category": 1
    },
    "index_goal": "Índice sobre vendedor para navegación de catálogo por seller."
})

# Q04 - Búsqueda textual inicial con regex
critical_queries.append({
    "code": "Q04",
    "collection": "product_catalog",
    "operation": "find",
    "business_purpose": "Buscar productos por texto en nombre o palabras clave.",
    "filter": {
        "$or": [
            {"name": {"$regex": "product", "$options": "i"}},
            {"search_keywords": {"$regex": "product", "$options": "i"}}
        ]
    },
    "sort": [("name", 1)],
    "projection": {
        "_id": 0,
        "product_id": 1,
        "name": 1,
        "search_keywords": 1,
        "category": 1
    },
    "index_goal": "Comparar búsqueda con regex frente a índice de texto en hitos posteriores."
})

# Q05 - Reseñas por producto
critical_queries.append({
    "code": "Q05",
    "collection": "product_reviews",
    "operation": "find",
    "business_purpose": "Consultar reseñas asociadas a un producto.",
    "filter": {"product_id": selected_product_id} if selected_product_id else {},
    "sort": [(review_date_field, -1)] if review_date_field else [("_id", 1)],
    "projection": {
        "_id": 0,
        "product_id": 1,
        "review_id": 1,
        "review_score": 1,
        "review_comment_message": 1,
        "created_at": 1
    },
    "index_goal": "Índice por product_id y fecha de reseña."
})

# Q06 - Reseñas críticas o de baja calificación
q06_filter = {}
if review_score_field:
    q06_filter[review_score_field] = {"$lte": 2}

critical_queries.append({
    "code": "Q06",
    "collection": "product_reviews",
    "operation": "find",
    "business_purpose": "Identificar reseñas de baja calificación para análisis de satisfacción.",
    "filter": q06_filter,
    "sort": [(review_score_field, 1)] if review_score_field else [("_id", 1)],
    "projection": {
        "_id": 0,
        "product_id": 1,
        "review_id": 1,
        "review_score": 1,
        "review_comment_message": 1
    },
    "index_goal": "Índice parcial para subconjunto de reseñas críticas."
})

# Q07 - Consulta analítica por categoría
q07_pipeline = []

if category_field:
    q07_pipeline.append({"$match": {category_field: {"$ne": None}}})

q07_pipeline.extend([
    {
        "$group": {
            "_id": f"${category_field}" if category_field else "$category",
            "total_products": {"$sum": 1}
        }
    },
    {"$sort": {"total_products": -1}},
    {"$limit": 10}
])

critical_queries.append({
    "code": "Q07",
    "collection": "product_catalog",
    "operation": "aggregate",
    "business_purpose": "Analizar concentración de productos por categoría.",
    "pipeline": q07_pipeline,
    "index_goal": "Soporte para agregaciones por categoría y diseño posterior de sharding."
})

critical_queries_df = pd.DataFrame([
    {
        "code": q["code"],
        "collection": q["collection"],
        "operation": q["operation"],
        "business_purpose": q["business_purpose"],
        "index_goal": q["index_goal"]
    }
    for q in critical_queries
])

critical_queries_df

,code,collection,operation,business_purpose,index_goal
0,Q01,product_catalog,find,"Consultar productos por categoría, ordenados p...",Índice compuesto para categoría y ordenamiento.
1,Q02,product_catalog,find,Filtrar productos por categoría y precio mínimo.,Índice compuesto aplicando regla ESR: igualdad...
2,Q03,product_catalog,find,Consultar productos asociados a un vendedor.,Índice sobre vendedor para navegación de catál...
3,Q04,product_catalog,find,Buscar productos por texto en nombre o palabra...,Comparar búsqueda con regex frente a índice de...
4,Q05,product_reviews,find,Consultar reseñas asociadas a un producto.,Índice por product_id y fecha de reseña.
5,Q06,product_reviews,find,Identificar reseñas de baja calificación para ...,Índice parcial para subconjunto de reseñas crí...
6,Q07,product_catalog,aggregate,Analizar concentración de productos por catego...,Soporte para agregaciones por categoría y dise...


**2.6 Validar que las consultas retornen resultados**

Esta celda valida que las consultas críticas sean ejecutables y retornen resultados o, al menos, que no fallen por nombres de campos incorrectos. Todavía no se evalúa rendimiento; eso se hará en el Hito 3 con explain("executionStats"), donde nos interesarán métricas como documentos examinados, llaves examinadas y tiempo de ejecución.

In [ ]:
# Hito 2.6 - Validación funcional de consultas críticas

def get_collection(collection_name):
    return db[collection_name]

validation_results = []

for query in critical_queries:
    collection = get_collection(query["collection"])

    try:
        if query["operation"] == "find":
            cursor = collection.find(
                query["filter"],
                query.get("projection", None)
            )

            if query.get("sort"):
                cursor = cursor.sort(query["sort"])

            preview = list(cursor.limit(3))
            result_count = collection.count_documents(query["filter"])

        elif query["operation"] == "aggregate":
            preview = list(collection.aggregate(query["pipeline"]))
            result_count = len(preview)

        else:
            preview = []
            result_count = 0

        validation_results.append({
            "code": query["code"],
            "collection": query["collection"],
            "operation": query["operation"],
            "status": "OK",
            "result_count": result_count,
            "preview_rows": len(preview),
            "error": None
        })

    except Exception as e:
        validation_results.append({
            "code": query["code"],
            "collection": query["collection"],
            "operation": query["operation"],
            "status": "ERROR",
            "result_count": None,
            "preview_rows": None,
            "error": str(e)
        })

validation_results_df = pd.DataFrame(validation_results)
validation_results_df

,code,collection,operation,status,result_count,preview_rows,error
0,Q01,product_catalog,find,OK,2,2,None
1,Q02,product_catalog,find,OK,1,1,None
2,Q03,product_catalog,find,OK,1,1,None
3,Q04,product_catalog,find,OK,1000,3,None
4,Q05,product_reviews,find,OK,1,1,None
5,Q06,product_reviews,find,OK,15275,3,None
6,Q07,product_catalog,aggregate,OK,10,10,None


**2.7 Ver una muestra de resultados por consulta**

Esta celda imprime una muestra pequeña de cada consulta. La idea es dejar evidencia visual en el Colab de que las consultas críticas están bien planteadas desde el punto de vista funcional.

In [ ]:
# Hito 2.7 - Muestra rápida de resultados por consulta

for query in critical_queries:
    print("=" * 80)
    print(f"{query['code']} - {query['business_purpose']}")
    print("Colección:", query["collection"])
    print("Operación:", query["operation"])

    collection = get_collection(query["collection"])

    if query["operation"] == "find":
        cursor = collection.find(
            query["filter"],
            query.get("projection", None)
        )

        if query.get("sort"):
            cursor = cursor.sort(query["sort"])

        preview = list(cursor.limit(2))
        display(pd.DataFrame(preview))

    elif query["operation"] == "aggregate":
        preview = list(collection.aggregate(query["pipeline"]))
        display(pd.DataFrame(preview))

Q01 - Consultar productos por categoría, ordenados por relevancia comercial.
Colección: product_catalog
Operación: find


,product_id,name,category,price_summary,rating_summary,sales_summary
0,07f01b6fcacc1b187a71e5074199db2d,Product agro_industry_and_commerce 07f01b6f,"{'name': 'agro_industria_e_comercio', 'name_tr...","{'min_price': 57.0, 'max_price': 57.0, 'avg_pr...","{'average_rating': 5.0, 'total_reviews': 1, 'l...","{'total_units_sold': 1, 'total_orders': 1, 'la..."
1,613d093272cb8f74f25a01e430155a6a,Product agro_industry_and_commerce 613d0932,"{'name': 'agro_industria_e_comercio', 'name_tr...","{'min_price': 29.5, 'max_price': 29.5, 'avg_pr...","{'average_rating': 5.0, 'total_reviews': 1, 'l...","{'total_units_sold': 1, 'total_orders': 1, 'la..."


Q02 - Filtrar productos por categoría y precio mínimo.
Colección: product_catalog
Operación: find


,product_id,name,category,price_summary
0,07f01b6fcacc1b187a71e5074199db2d,Product agro_industry_and_commerce 07f01b6f,"{'name': 'agro_industria_e_comercio', 'name_tr...","{'min_price': 57.0, 'max_price': 57.0, 'avg_pr..."


Q03 - Consultar productos asociados a un vendedor.
Colección: product_catalog
Operación: find


...

,review_id,product_id,created_at
0,d71da8fd8c6e3adef26be965f065b8a1,1e9e8ef04dbcff4541ed26657ea517e5,2026-06-01 20:50:07.727


Q06 - Identificar reseñas de baja calificación para análisis de satisfacción.
Colección: product_reviews
Operación: find


,review_id,product_id
0,efe49f1d6f951dd88b51e6ccd4cc548f,8fbd36d3b045f5f38b252b1513478f38
1,6cf47345d15e054dd6df872e929bdb27,d8342ac8b27a03e8d25280c296dfa830


Q07 - Analizar concentración de productos por categoría.
Colección: product_catalog
Operación: aggregate


,_id,total_products
0,bed_bath_table,86
1,sports_leisure,83
2,health_beauty,75
3,furniture_decor,72
4,housewares,64
5,auto,57
6,computers_accessories,45
7,toys,45
8,telephony,38
9,watches_gifts,36


# Hito 3 — Medición base con .explain("executionStats")

**3.1 Funciones auxiliares para ejecutar explain**

Las funciones permiten ejecutar y resumir los planes de ejecución. Como MongoDB puede devolver planes con estructuras diferentes para find y aggregate, se hace una búsqueda recursiva de métricas clave como tiempo, documentos examinados, llaves examinadas, documentos retornados, índices usados y etapas del plan.

In [ ]:
# Hito 3.1 - Funciones auxiliares para ejecutar explain("executionStats")

import json
import pandas as pd
from datetime import datetime

def make_json_safe(value):
    """
    Convierte valores especiales de MongoDB/Python a texto para poder exportarlos o mostrarlos.
    """
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def recursive_find_keys(obj, target_key):
    """
    Busca recursivamente valores asociados a una llave dentro de un diccionario/lista.
    """
    results = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == target_key:
                results.append(value)
            results.extend(recursive_find_keys(value, target_key))

    elif isinstance(obj, list):
        for item in obj:
            results.extend(recursive_find_keys(item, target_key))

    return results


def recursive_find_stage_names(obj):
    """
    Identifica nombres de etapas relevantes dentro del plan de ejecución.
    """
    stages = []

    if isinstance(obj, dict):
        if "stage" in obj:
            stages.append(obj["stage"])

        for value in obj.values():
            stages.extend(recursive_find_stage_names(value))

    elif isinstance(obj, list):
        for item in obj:
            stages.extend(recursive_find_stage_names(item))

    return stages


def summarize_explain(query_code, collection_name, operation, explain_result):
    """
    Extrae métricas principales del resultado de explain.
    """
    execution_times = recursive_find_keys(explain_result, "executionTimeMillis")
    docs_examined = recursive_find_keys(explain_result, "totalDocsExamined")
    keys_examined = recursive_find_keys(explain_result, "totalKeysExamined")
    n_returned = recursive_find_keys(explain_result, "nReturned")
    index_names = recursive_find_keys(explain_result, "indexName")
    stage_names = recursive_find_stage_names(explain_result)

    return {
        "query_code": query_code,
        "collection": collection_name,
        "operation": operation,
        "execution_time_ms": max(execution_times) if execution_times else None,
        "total_docs_examined": max(docs_examined) if docs_examined else None,
        "total_keys_examined": max(keys_examined) if keys_examined else None,
        "n_returned": max(n_returned) if n_returned else None,
        "indexes_used": sorted(list(set(index_names))) if index_names else [],
        "stages_detected": sorted(list(set(stage_names))) if stage_names else [],
        "captured_at": datetime.now().isoformat(timespec="seconds")
    }

**3.2 Ejecutar línea base de consultas find**

Esta celda ejecuta explain para las consultas tipo find definidas en el Hito 2. Aquí todavía no se optimiza nada; solo se captura cómo MongoDB resuelve cada consulta en su estado inicial.

In [ ]:
# Hito 3.2 - Ejecución de explain para consultas find

baseline_explains = {}
baseline_summary = []

for query in critical_queries:
    if query["operation"] != "find":
        continue

    collection = db[query["collection"]]

    explain_command = {
        "find": query["collection"],
        "filter": query.get("filter", {}),
    }

    if query.get("projection"):
        explain_command["projection"] = query["projection"]

    if query.get("sort"):
        explain_command["sort"] = dict(query["sort"])

    explain_result = db.command(
        "explain",
        explain_command,
        verbosity="executionStats"
    )

    baseline_explains[query["code"]] = explain_result

    baseline_summary.append(
        summarize_explain(
            query_code=query["code"],
            collection_name=query["collection"],
            operation=query["operation"],
            explain_result=explain_result
        )
    )

baseline_find_df = pd.DataFrame(baseline_summary)
baseline_find_df

,query_code,collection,operation,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,stages_detected,captured_at
0,Q01,product_catalog,find,1,2,2,2,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T19:32:45
1,Q02,product_catalog,find,0,1,1,1,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T19:32:45
2,Q03,product_catalog,find,0,1,1,1,"[idx_catalog_seller_name, idx_seller_summary_s...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T19:32:45
3,Q04,product_catalog,find,4,1000,0,1000,[],"[COLLSCAN, PROJECTION_SIMPLE, SORT, SUBPLAN]",2026-06-15T19:32:46
4,Q05,product_reviews,find,0,1,1,1,"[idx_reviews_product_date, idx_reviews_product...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T19:32:46
5,Q06,product_reviews,find,42,15275,15275,15275,"[idx_reviews_low_score_partial, idx_reviews_sc...","[FETCH, IXSCAN, PROJECTION_SIMPLE]",2026-06-15T19:32:46


**3.3 Ejecutar línea base de consultas aggregate**

Esta celda mide las consultas tipo aggregate, incluyendo el pipeline definido en el Hito 2. Se activa allowDiskUse para permitir que MongoDB use disco si el pipeline supera límites de memoria, aunque en esta etapa solo estamos midiendo la línea base.

In [ ]:
# Hito 3.3 - Ejecución de explain para consultas aggregate

for query in critical_queries:
    if query["operation"] != "aggregate":
        continue

    explain_command = {
        "aggregate": query["collection"],
        "pipeline": query["pipeline"],
        "cursor": {},
        "allowDiskUse": True
    }

    explain_result = db.command(
        "explain",
        explain_command,
        verbosity="executionStats"
    )

    baseline_explains[query["code"]] = explain_result

    baseline_summary.append(
        summarize_explain(
            query_code=query["code"],
            collection_name=query["collection"],
            operation=query["operation"],
            explain_result=explain_result
        )
    )

baseline_df = pd.DataFrame(baseline_summary)
baseline_df

,query_code,collection,operation,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,stages_detected,captured_at
0,Q01,product_catalog,find,1,2,2,2,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T19:32:45
1,Q02,product_catalog,find,0,1,1,1,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T19:32:45
2,Q03,product_catalog,find,0,1,1,1,"[idx_catalog_seller_name, idx_seller_summary_s...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T19:32:45
3,Q04,product_catalog,find,4,1000,0,1000,[],"[COLLSCAN, PROJECTION_SIMPLE, SORT, SUBPLAN]",2026-06-15T19:32:46
4,Q05,product_reviews,find,0,1,1,1,"[idx_reviews_product_date, idx_reviews_product...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T19:32:46
5,Q06,product_reviews,find,42,15275,15275,15275,"[idx_reviews_low_score_partial, idx_reviews_sc...","[FETCH, IXSCAN, PROJECTION_SIMPLE]",2026-06-15T19:32:46
6,Q07,product_catalog,aggregate,4,0,972,971,"[idx_catalog_category_price, idx_catalog_categ...","[GROUP, IXSCAN, PROJECTION_DEFAULT, group, ixs...",2026-06-15T19:32:46


**3.4 Clasificar el tipo de plan detectado**

Esta celda clasifica de forma simple cada consulta según el tipo de plan detectado. Si aparece COLLSCAN, MongoDB está revisando muchos documentos sin apoyo suficiente de índices. Si aparece IXSCAN, ya existe algún índice participando en la consulta

In [ ]:
# Hito 3.4 - Clasificación simple del plan de ejecución

def classify_plan(stages):
    stages_text = " ".join(stages)

    if "COLLSCAN" in stages_text:
        return "COLLSCAN - Revisión completa de colección"
    elif "IXSCAN" in stages_text:
        return "IXSCAN - Uso de índice"
    elif "TEXT_MATCH" in stages_text:
        return "TEXT_MATCH - Uso de índice de texto"
    else:
        return "OTRO - Revisar plan detallado"


baseline_df["plan_classification"] = baseline_df["stages_detected"].apply(classify_plan)

baseline_df[
    [
        "query_code",
        "collection",
        "operation",
        "execution_time_ms",
        "total_docs_examined",
        "total_keys_examined",
        "n_returned",
        "indexes_used",
        "plan_classification"
    ]
]

,query_code,collection,operation,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,plan_classification
0,Q01,product_catalog,find,1,2,2,2,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice
1,Q02,product_catalog,find,0,1,1,1,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice
2,Q03,product_catalog,find,0,1,1,1,"[idx_catalog_seller_name, idx_seller_summary_s...",IXSCAN - Uso de índice
3,Q04,product_catalog,find,4,1000,0,1000,[],COLLSCAN - Revisión completa de colección
4,Q05,product_reviews,find,0,1,1,1,"[idx_reviews_product_date, idx_reviews_product...",IXSCAN - Uso de índice
5,Q06,product_reviews,find,42,15275,15275,15275,"[idx_reviews_low_score_partial, idx_reviews_sc...",IXSCAN - Uso de índice
6,Q07,product_catalog,aggregate,4,0,972,971,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice


**3.5 Calcular eficiencia de lectura**

Esta celda calcula una métrica sencilla: cuántos documentos o llaves examina MongoDB por cada resultado retornado. Mientras más alto sea docs_per_result, más costosa puede ser la consulta y más candidata será para optimización.

In [ ]:
# Hito 3.5 - Cálculo de eficiencia base

baseline_df["docs_per_result"] = baseline_df.apply(
    lambda row: (
        row["total_docs_examined"] / row["n_returned"]
        if row["n_returned"] not in [None, 0] and row["total_docs_examined"] is not None
        else None
    ),
    axis=1
)

baseline_df["keys_per_result"] = baseline_df.apply(
    lambda row: (
        row["total_keys_examined"] / row["n_returned"]
        if row["n_returned"] not in [None, 0] and row["total_keys_examined"] is not None
        else None
    ),
    axis=1
)

baseline_efficiency_df = baseline_df[
    [
        "query_code",
        "collection",
        "execution_time_ms",
        "total_docs_examined",
        "total_keys_examined",
        "n_returned",
        "docs_per_result",
        "keys_per_result",
        "plan_classification"
    ]
]

baseline_efficiency_df

,query_code,collection,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,docs_per_result,keys_per_result,plan_classification
0,Q01,product_catalog,1,2,2,2,1.0,1.00000,IXSCAN - Uso de índice
1,Q02,product_catalog,0,1,1,1,1.0,1.00000,IXSCAN - Uso de índice
2,Q03,product_catalog,0,1,1,1,1.0,1.00000,IXSCAN - Uso de índice
3,Q04,product_catalog,4,1000,0,1000,1.0,0.00000,COLLSCAN - Revisión completa de colección
4,Q05,product_reviews,0,1,1,1,1.0,1.00000,IXSCAN - Uso de índice
5,Q06,product_reviews,42,15275,15275,15275,1.0,1.00000,IXSCAN - Uso de índice
6,Q07,product_catalog,4,0,972,971,0.0,1.00103,IXSCAN - Uso de índice


**3.6 Guardar evidencia de línea base en CSV**

Esta celda guarda la tabla de línea base en un archivo CSV. Más adelante se usará para comparar contra las métricas posteriores a los índices y para documentar resultados en el Word final.

In [ ]:
# Hito 3.6 - Exportar resultados base para evidencia

baseline_df.to_csv("hito3_baseline_explain_summary.csv", index=False)

print("Archivo generado: hito3_baseline_explain_summary.csv")

Archivo generado: hito3_baseline_explain_summary.csv


**3.7 Ver detalle de un plan específico**

Esta celda permite inspeccionar el plan completo de una consulta específica. Es útil si alguna consulta aparece como lenta, con COLLSCAN, o con muchos documentos examinados.

In [ ]:
# Hito 3.7 - Visualización opcional de un plan específico

query_to_inspect = "Q01"

print(json.dumps(
    baseline_explains[query_to_inspect],
    indent=2,
    default=str
))

{
  "explainVersion": "1",
  "queryPlanner": {
    "namespace": "ecommify_mongodb.product_catalog",
    "parsedQuery": {
      "category.name_translated": {
        "$eq": "agro_industry_and_commerce"
      }
    },
    "indexFilterSet": false,
    "queryHash": "1A1A91B6",
    "planCacheShapeHash": "1A1A91B6",
    "planCacheKey": "D8A40DBB",
    "optimizationTimeMillis": 1,
    "maxIndexedOrSolutionsReached": false,
    "maxIndexedAndSolutionsReached": false,
    "maxScansToExplodeReached": false,
    "prunedSimilarIndexes": false,
    "winningPlan": {
      "isCached": false,
      "stage": "PROJECTION_SIMPLE",
      "transformBy": {
        "_id": 0,
        "product_id": 1,
        "name": 1,
        "category": 1,
        "price_summary": 1,
        "sales_summary": 1,
        "rating_summary": 1
      },
      "inputStage": {
        "stage": "FETCH",
        "inputStage": {
          "stage": "IXSCAN",
          "keyPattern": {
            "category.name_translated": 1,
         

# Hito 4 — Creación de índices especializados
**4.1 Importar constantes de indexación**

esta celda importa las constantes que usa PyMongo para crear índices ascendentes, descendentes y de texto. Estos índices se aplicarán sobre las consultas críticas definidas en el Hito 2 y medidas en el Hito 3.

In [ ]:
# Hito 4.1 - Importación de constantes para crear índices

from pymongo import ASCENDING, DESCENDING, TEXT
import pandas as pd
from datetime import datetime

**4.2 Preparar función segura para crear índices**

Esta función crea índices de forma controlada y deja un registro de qué se intentó crear, en qué colección, con qué propósito y si la operación fue exitosa. Esto nos servirá como evidencia para el documento final.

In [ ]:
# Hito 4.2 - Función auxiliar para crear índices sin duplicar lógica

created_indexes_log = []

def create_index_safely(collection, keys, name, index_type, business_purpose, **options):
    """
    Crea un índice en MongoDB y registra el resultado.
    Si el índice ya existe con el mismo nombre y definición, MongoDB lo reutiliza.
    """
    try:
        index_name = collection.create_index(keys, name=name, **options)

        created_indexes_log.append({
            "collection": collection.name,
            "index_name": index_name,
            "keys": keys,
            "index_type": index_type,
            "business_purpose": business_purpose,
            "status": "OK",
            "error": None,
            "created_at": datetime.now().isoformat(timespec="seconds")
        })

        print(f"OK - Índice creado o existente: {collection.name}.{index_name}")

    except Exception as e:
        created_indexes_log.append({
            "collection": collection.name,
            "index_name": name,
            "keys": keys,
            "index_type": index_type,
            "business_purpose": business_purpose,
            "status": "ERROR",
            "error": str(e),
            "created_at": datetime.now().isoformat(timespec="seconds")
        })

        print(f"ERROR - No se pudo crear el índice {collection.name}.{name}")
        print(str(e))

**4.3 Crear índice compuesto para Q01**

Este índice apoya la consulta Q01, donde se navegan productos por categoría y se ordenan por relevancia comercial o ventas. La categoría funciona como filtro de igualdad y las ventas como criterio de ordenamiento.

In [ ]:
# Hito 4.3 - Índice compuesto para navegación por categoría y ordenamiento comercial

if category_field and sales_field:
    create_index_safely(
        collection=product_catalog,
        keys=[
            (category_field, ASCENDING),
            (sales_field, DESCENDING)
        ],
        name="idx_catalog_category_sales",
        index_type="compound",
        business_purpose="Optimizar Q01: productos por categoría ordenados por ventas."
    )
else:
    print("No se creó idx_catalog_category_sales porque falta category_field o sales_field.")

OK - Índice creado o existente: product_catalog.idx_catalog_category_sales


**4.4 Crear índice compuesto ESR para Q02**

Este índice aplica la lógica ESR: primero el campo de igualdad (category) y luego el campo usado para rango u ordenamiento (price). Está pensado para mejorar consultas de catálogo filtradas por categoría y precio.

In [ ]:
# Hito 4.4 - Índice compuesto ESR para categoría y precio

if category_field and price_field:
    create_index_safely(
        collection=product_catalog,
        keys=[
            (category_field, ASCENDING),
            (price_field, ASCENDING)
        ],
        name="idx_catalog_category_price",
        index_type="compound_esr",
        business_purpose="Optimizar Q02: filtro por categoría y rango/orden de precio."
    )
else:
    print("No se creó idx_catalog_category_price porque falta category_field o price_field.")

OK - Índice creado o existente: product_catalog.idx_catalog_category_price


**4.5 Crear índice para Q03 por vendedor**

Este índice busca mejorar la consulta de productos por vendedor. Se agrega name como segundo campo para apoyar el ordenamiento alfabético cuando se listan productos asociados a un mismo seller.

In [ ]:
# Hito 4.5 - Índice para consulta de productos por vendedor

create_index_safely(
    collection=product_catalog,
    keys=[
        ("seller_summary.seller_id", ASCENDING),
        ("name", ASCENDING)
    ],
    name="idx_catalog_seller_name",
    index_type="compound",
    business_purpose="Optimizar Q03: consulta de productos asociados a un vendedor."
)

OK - Índice creado o existente: product_catalog.idx_catalog_seller_name


**4.6 Crear índice de texto para Q04**

Esta celda primero revisa si ya existe un índice de texto en product_catalog. Si existe, no intenta crear otro y lo registra como REUSED. Si no existe, entonces sí crea el índice propuesto.

In [ ]:
# Hito 4.6 - Validación o reutilización de índice de texto para búsqueda

existing_text_index = None

for index in product_catalog.list_indexes():
    index_keys = dict(index.get("key", {}))

    # Los índices de texto aparecen internamente como _fts: "text"
    if index_keys.get("_fts") == "text":
        existing_text_index = index
        break

if existing_text_index:
    created_indexes_log.append({
        "collection": product_catalog.name,
        "index_name": existing_text_index.get("name"),
        "keys": dict(existing_text_index.get("key", {})),
        "index_type": "text_existing",
        "business_purpose": (
            "Reutilizar índice de texto existente para Q04: búsqueda textual "
            "en nombre, palabras clave y categoría."
        ),
        "status": "REUSED",
        "error": None,
        "created_at": datetime.now().isoformat(timespec="seconds")
    })

    print("OK - Se reutiliza índice de texto existente:")
    print("Nombre:", existing_text_index.get("name"))
    print("Weights:", existing_text_index.get("weights"))

else:
    create_index_safely(
        collection=product_catalog,
        keys=[
            ("name", TEXT),
            ("search_keywords", TEXT)
        ],
        name="idx_catalog_text_search",
        index_type="text",
        business_purpose="Optimizar Q04: búsqueda textual en nombre y palabras clave."
    )

OK - Se reutiliza índice de texto existente:
Nombre: idx_text_product_search
Weights: SON([('category.name_translated', 1), ('name', 1), ('search_keywords', 1)])


**4.7 Crear índice para Q05 en reseñas por producto**

Este índice apoya la consulta de reseñas asociadas a un producto. Si existe campo de fecha, también ayuda a devolver las reseñas más recientes primero.

In [ ]:
# Hito 4.7 - Índice para reseñas por producto y fecha

if review_date_field:
    create_index_safely(
        collection=product_reviews,
        keys=[
            ("product_id", ASCENDING),
            (review_date_field, DESCENDING)
        ],
        name="idx_reviews_product_date",
        index_type="compound",
        business_purpose="Optimizar Q05: consulta de reseñas por producto ordenadas por fecha."
    )
else:
    create_index_safely(
        collection=product_reviews,
        keys=[
            ("product_id", ASCENDING)
        ],
        name="idx_reviews_product",
        index_type="single_field",
        business_purpose="Optimizar Q05: consulta de reseñas por producto."
    )

OK - Índice creado o existente: product_reviews.idx_reviews_product_date


**4.8 Crear índice parcial para Q06**

Este índice parcial solo cubre reseñas con calificación baja. Es útil porque no indexa toda la colección, sino únicamente el subconjunto relevante para análisis de satisfacción o alertas de calidad.

In [ ]:
# Hito 4.8 - Índice parcial para reseñas críticas o de baja calificación

if review_score_field:
    create_index_safely(
        collection=product_reviews,
        keys=[
            (review_score_field, ASCENDING),
            ("product_id", ASCENDING)
        ],
        name="idx_reviews_low_score_partial",
        index_type="partial",
        business_purpose="Optimizar Q06: reseñas con baja calificación.",
        partialFilterExpression={
            review_score_field: {"$lte": 2}
        }
    )
else:
    print("No se creó idx_reviews_low_score_partial porque falta review_score_field.")

OK - Índice creado o existente: product_reviews.idx_reviews_low_score_partial


**4.9 Crear índice para agregación por categoría Q07**

Esta celda primero revisa si ya existe un índice sobre el campo de categoría seleccionado. Si existe, lo registra como reutilizado (REUSED) y evita crear un índice duplicado. Si no existe, crea el índice propuesto para apoyar la consulta analítica Q07.

In [ ]:
# Hito 4.9 - Validación o reutilización de índice para agregación por categoría

existing_category_index = None

if category_field:
    for index in product_catalog.list_indexes():
        index_keys = dict(index.get("key", {}))

        # Busca un índice que tenga exactamente el campo de categoría como primer campo
        if category_field in index_keys:
            existing_category_index = index
            break

    if existing_category_index:
        created_indexes_log.append({
            "collection": product_catalog.name,
            "index_name": existing_category_index.get("name"),
            "keys": dict(existing_category_index.get("key", {})),
            "index_type": "single_field_existing",
            "business_purpose": (
                "Reutilizar índice existente para Q07: agregación y análisis "
                "por categoría."
            ),
            "status": "REUSED",
            "error": None,
            "created_at": datetime.now().isoformat(timespec="seconds")
        })

        print("OK - Se reutiliza índice existente para categoría:")
        print("Nombre:", existing_category_index.get("name"))
        print("Keys:", dict(existing_category_index.get("key", {})))

    else:
        create_index_safely(
            collection=product_catalog,
            keys=[
                (category_field, ASCENDING)
            ],
            name="idx_catalog_category",
            index_type="single_field",
            business_purpose="Optimizar Q07: agregación y análisis por categoría."
        )
else:
    print("No se creó idx_catalog_category porque falta category_field.")

OK - Se reutiliza índice existente para categoría:
Nombre: idx_category_name_translated
Keys: {'category.name_translated': 1}


**4.10 Ver resumen de índices creados**

Esta celda consolida los índices creados durante el Hito 4. La tabla muestra nombre, colección, tipo de índice, propósito de negocio y estado de creación.

In [ ]:
# Hito 4.10 - Resumen de índices creados en este hito

created_indexes_df = pd.DataFrame(created_indexes_log)
created_indexes_df

,collection,index_name,keys,index_type,business_purpose,status,error,created_at
0,product_catalog,idx_catalog_category_sales,"[(category.name_translated, 1), (sales_summary...",compound,Optimizar Q01: productos por categoría ordenad...,OK,None,2026-06-15T19:33:03
1,product_catalog,idx_catalog_category_price,"[(category.name_translated, 1), (price_summary...",compound_esr,Optimizar Q02: filtro por categoría y rango/or...,OK,None,2026-06-15T19:33:03
2,product_catalog,idx_catalog_seller_name,"[(seller_summary.seller_id, 1), (name, 1)]",compound,Optimizar Q03: consulta de productos asociados...,OK,None,2026-06-15T19:33:04
3,product_catalog,idx_text_product_search,"{'_fts': 'text', '_ftsx': 1}",text_existing,Reutilizar índice de texto existente para Q04:...,REUSED,None,2026-06-15T19:33:04
4,product_reviews,idx_reviews_product_date,"[(product_id, 1), (created_at, -1)]",compound,Optimizar Q05: consulta de reseñas por product...,OK,None,2026-06-15T19:33:04
5,product_reviews,idx_reviews_low_score_partial,"[(score, 1), (product_id, 1)]",partial,Optimizar Q06: reseñas con baja calificación.,OK,None,2026-06-15T19:33:04
6,product_catalog,idx_category_name_translated,{'category.name_translated': 1},single_field_existing,Reutilizar índice existente para Q07: agregaci...,REUSED,None,2026-06-15T19:33:04


**4.11 Listar índices actuales de las colecciones**

Esta celda lista todos los índices disponibles después de la optimización. Sirve para verificar que los índices del Hito 4 quedaron registrados en MongoDB Atlas.

In [ ]:
# Hito 4.11 - Listado final de índices existentes en las colecciones objetivo

index_inventory = []

for collection in [product_catalog, product_reviews]:
    for index in collection.list_indexes():
        index_inventory.append({
            "collection": collection.name,
            "index_name": index.get("name"),
            "keys": dict(index.get("key", {})),
            "unique": index.get("unique", False),
            "partial_filter": index.get("partialFilterExpression", None),
            "weights": index.get("weights", None)
        })

index_inventory_df = pd.DataFrame(index_inventory)
index_inventory_df

,collection,index_name,keys,unique,partial_filter,weights
0,product_catalog,_id_,{'_id': 1},False,None,None
1,product_catalog,idx_category_name_translated,{'category.name_translated': 1},False,None,None
2,product_catalog,idx_total_units_sold,{'sales_summary.total_units_sold': -1},False,None,None
3,product_catalog,idx_average_rating,{'rating_summary.average_rating': -1},False,None,None
4,product_catalog,idx_seller_summary_seller_id,{'seller_summary.seller_id': 1},False,None,None
5,product_catalog,idx_category_rating,"{'category.name_translated': 1, 'rating_summar...",False,None,None
6,product_catalog,idx_text_product_search,"{'_fts': 'text', '_ftsx': 1}",False,None,"{'category.name_translated': 1, 'name': 1, 'se..."
7,product_catalog,idx_catalog_category_sales,"{'category.name_translated': 1, 'sales_summary...",False,None,None
8,product_catalog,idx_catalog_category_price,"{'category.name_translated': 1, 'price_summary...",False,None,None
9,product_catalog,idx_catalog_seller_name,"{'seller_summary.seller_id': 1, 'name': 1}",False,None,None


**4.12 Guardar evidencia del Hito 4**

# Hito 4.12 - Exportar evidencias de índices

created_indexes_df.to_csv("hito4_created_indexes.csv", index=False)
index_inventory_df.to_csv("hito4_index_inventory.csv", index=False)

print("Archivos generados:")
print("- hito4_created_indexes.csv")
print("- hito4_index_inventory.csv")

In [ ]:
# Hito 4.12 - Exportar evidencias de índices

created_indexes_df.to_csv("hito4_created_indexes.csv", index=False)
index_inventory_df.to_csv("hito4_index_inventory.csv", index=False)

print("Archivos generados:")
print("- hito4_created_indexes.csv")
print("- hito4_index_inventory.csv")

Archivos generados:
- hito4_created_indexes.csv
- hito4_index_inventory.csv


# Hito 5 — Medición posterior y comparación
**5.1 Preparar consultas optimizadas**

En Q04 se hará un ajuste: la consulta original usaba $regex, pero para aprovechar el índice de texto reutilizado se debe usar $text.

esta celda conserva las consultas Q01-Q07, pero cambia Q04 para que use $text en lugar de $regex. Esto permite que MongoDB aproveche el índice de texto idx_text_product_search.

In [ ]:
# Hito 5.1 - Preparar versión optimizada de las consultas críticas

import copy
import pandas as pd
from datetime import datetime

optimized_queries = copy.deepcopy(critical_queries)

for query in optimized_queries:
    if query["code"] == "Q04":
        query["filter"] = {
            "$text": {
                "$search": "product"
            }
        }
        query["projection"] = {
            "_id": 0,
            "product_id": 1,
            "name": 1,
            "category": 1,
            "search_keywords": 1,
            "text_score": {
                "$meta": "textScore"
            }
        }
        query["sort"] = [
            ("text_score", {"$meta": "textScore"})
        ]
        query["business_purpose"] = (
            "Buscar productos por texto usando índice full-text existente."
        )

print("Consultas optimizadas preparadas.")

Consultas optimizadas preparadas.


**5.2 Ejecutar medición posterior para consultas find**

Esta celda ejecuta nuevamente los planes de ejecución para las consultas tipo find, ahora después de haber creado o reutilizado índices. Todavía no interpreta los resultados; solo captura las métricas posteriores.

In [ ]:
# Hito 5.2 - Ejecutar explain posterior para consultas find

optimized_explains = {}
optimized_summary = []

for query in optimized_queries:
    if query["operation"] != "find":
        continue

    explain_command = {
        "find": query["collection"],
        "filter": query.get("filter", {})
    }

    if query.get("projection"):
        explain_command["projection"] = query["projection"]

    if query.get("sort"):
        explain_command["sort"] = dict(query["sort"])

    explain_result = db.command(
        "explain",
        explain_command,
        verbosity="executionStats"
    )

    optimized_explains[query["code"]] = explain_result

    optimized_summary.append(
        summarize_explain(
            query_code=query["code"],
            collection_name=query["collection"],
            operation=query["operation"],
            explain_result=explain_result
        )
    )

optimized_find_df = pd.DataFrame(optimized_summary)
optimized_find_df

,query_code,collection,operation,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,stages_detected,captured_at
0,Q01,product_catalog,find,5,2,2,2,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T20:00:05
1,Q02,product_catalog,find,0,1,1,1,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T20:00:05
2,Q03,product_catalog,find,3,1,1,1,"[idx_catalog_seller_name, idx_seller_summary_s...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T20:00:05
3,Q04,product_catalog,find,20,1000,1000,1000,[idx_text_product_search],"[IXSCAN, PROJECTION_DEFAULT, SORT, TEXT_MATCH,...",2026-06-15T20:00:05
4,Q05,product_reviews,find,6,1,1,1,"[idx_reviews_product_date, idx_reviews_product...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T20:00:05
5,Q06,product_reviews,find,106,15275,15275,15275,"[idx_reviews_low_score_partial, idx_reviews_sc...","[FETCH, IXSCAN, PROJECTION_SIMPLE]",2026-06-15T20:00:05


**5.3 Ejecutar medición posterior para consultas aggregate**

Esta celda mide la consulta de agregación Q07 después de los índices. Se mantiene allowDiskUse=True como práctica segura para pipelines que puedan crecer en volumen.

In [ ]:
# Hito 5.3 - Ejecutar explain posterior para consultas aggregate

for query in optimized_queries:
    if query["operation"] != "aggregate":
        continue

    explain_command = {
        "aggregate": query["collection"],
        "pipeline": query["pipeline"],
        "cursor": {},
        "allowDiskUse": True
    }

    explain_result = db.command(
        "explain",
        explain_command,
        verbosity="executionStats"
    )

    optimized_explains[query["code"]] = explain_result

    optimized_summary.append(
        summarize_explain(
            query_code=query["code"],
            collection_name=query["collection"],
            operation=query["operation"],
            explain_result=explain_result
        )
    )

optimized_df = pd.DataFrame(optimized_summary)
optimized_df

,query_code,collection,operation,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,stages_detected,captured_at
0,Q01,product_catalog,find,5,2,2,2,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T20:00:05
1,Q02,product_catalog,find,0,1,1,1,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T20:00:05
2,Q03,product_catalog,find,3,1,1,1,"[idx_catalog_seller_name, idx_seller_summary_s...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T20:00:05
3,Q04,product_catalog,find,20,1000,1000,1000,[idx_text_product_search],"[IXSCAN, PROJECTION_DEFAULT, SORT, TEXT_MATCH,...",2026-06-15T20:00:05
4,Q05,product_reviews,find,6,1,1,1,"[idx_reviews_product_date, idx_reviews_product...","[FETCH, IXSCAN, PROJECTION_SIMPLE, SORT]",2026-06-15T20:00:05
5,Q06,product_reviews,find,106,15275,15275,15275,"[idx_reviews_low_score_partial, idx_reviews_sc...","[FETCH, IXSCAN, PROJECTION_SIMPLE]",2026-06-15T20:00:05
6,Q07,product_catalog,aggregate,3,0,972,971,"[idx_catalog_category_price, idx_catalog_categ...","[GROUP, IXSCAN, PROJECTION_DEFAULT, group, ixs...",2026-06-15T20:00:11


**5.4 Clasificar planes posteriores**

Esta celda clasifica los planes posteriores según el tipo de operación detectada. Lo ideal es ver más uso de IXSCAN o TEXT_MATCH y menos COLLSCAN.

In [ ]:
# Hito 5.4 - Clasificar planes posteriores

optimized_df["plan_classification"] = optimized_df["stages_detected"].apply(classify_plan)

optimized_df[
    [
        "query_code",
        "collection",
        "operation",
        "execution_time_ms",
        "total_docs_examined",
        "total_keys_examined",
        "n_returned",
        "indexes_used",
        "plan_classification"
    ]
]

,query_code,collection,operation,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,plan_classification
0,Q01,product_catalog,find,5,2,2,2,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice
1,Q02,product_catalog,find,0,1,1,1,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice
2,Q03,product_catalog,find,3,1,1,1,"[idx_catalog_seller_name, idx_seller_summary_s...",IXSCAN - Uso de índice
3,Q04,product_catalog,find,20,1000,1000,1000,[idx_text_product_search],IXSCAN - Uso de índice
4,Q05,product_reviews,find,6,1,1,1,"[idx_reviews_product_date, idx_reviews_product...",IXSCAN - Uso de índice
5,Q06,product_reviews,find,106,15275,15275,15275,"[idx_reviews_low_score_partial, idx_reviews_sc...",IXSCAN - Uso de índice
6,Q07,product_catalog,aggregate,3,0,972,971,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice


**5.5 Calcular eficiencia posterior**

Esta celda calcula la eficiencia posterior a la indexación. Nos interesa especialmente si bajan total_docs_examined y docs_per_result, porque eso indica que MongoDB examina menos documentos para producir el mismo resultado.

In [ ]:
# Hito 5.5 - Calcular eficiencia posterior

optimized_df["docs_per_result"] = optimized_df.apply(
    lambda row: (
        row["total_docs_examined"] / row["n_returned"]
        if row["n_returned"] not in [None, 0] and row["total_docs_examined"] is not None
        else None
    ),
    axis=1
)

optimized_df["keys_per_result"] = optimized_df.apply(
    lambda row: (
        row["total_keys_examined"] / row["n_returned"]
        if row["n_returned"] not in [None, 0] and row["total_keys_examined"] is not None
        else None
    ),
    axis=1
)

optimized_efficiency_df = optimized_df[
    [
        "query_code",
        "collection",
        "execution_time_ms",
        "total_docs_examined",
        "total_keys_examined",
        "n_returned",
        "docs_per_result",
        "keys_per_result",
        "indexes_used",
        "plan_classification"
    ]
]

optimized_efficiency_df

,query_code,collection,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,docs_per_result,keys_per_result,indexes_used,plan_classification
0,Q01,product_catalog,5,2,2,2,1.0,1.00000,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice
1,Q02,product_catalog,0,1,1,1,1.0,1.00000,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice
2,Q03,product_catalog,3,1,1,1,1.0,1.00000,"[idx_catalog_seller_name, idx_seller_summary_s...",IXSCAN - Uso de índice
3,Q04,product_catalog,20,1000,1000,1000,1.0,1.00000,[idx_text_product_search],IXSCAN - Uso de índice
4,Q05,product_reviews,6,1,1,1,1.0,1.00000,"[idx_reviews_product_date, idx_reviews_product...",IXSCAN - Uso de índice
5,Q06,product_reviews,106,15275,15275,15275,1.0,1.00000,"[idx_reviews_low_score_partial, idx_reviews_sc...",IXSCAN - Uso de índice
6,Q07,product_catalog,3,0,972,971,0.0,1.00103,"[idx_catalog_category_price, idx_catalog_categ...",IXSCAN - Uso de índice


**5.6 Comparar antes/después**

Esta celda compara las métricas antes y después. Las mejoras más importantes para documentar son reducción de documentos examinados, aparición de índices usados y reducción de tiempo de ejecución cuando sea visible.

In [ ]:
# Hito 5.6 - Comparación entre línea base y medición posterior

baseline_compare = baseline_df[
    [
        "query_code",
        "execution_time_ms",
        "total_docs_examined",
        "total_keys_examined",
        "n_returned",
        "plan_classification",
        "indexes_used"
    ]
].copy()

optimized_compare = optimized_df[
    [
        "query_code",
        "execution_time_ms",
        "total_docs_examined",
        "total_keys_examined",
        "n_returned",
        "plan_classification",
        "indexes_used"
    ]
].copy()

comparison_df = baseline_compare.merge(
    optimized_compare,
    on="query_code",
    suffixes=("_before", "_after")
)

comparison_df["execution_time_delta_ms"] = (
    comparison_df["execution_time_ms_before"] -
    comparison_df["execution_time_ms_after"]
)

comparison_df["docs_examined_delta"] = (
    comparison_df["total_docs_examined_before"] -
    comparison_df["total_docs_examined_after"]
)

comparison_df["keys_examined_delta"] = (
    comparison_df["total_keys_examined_before"] -
    comparison_df["total_keys_examined_after"]
)

comparison_df["execution_time_improvement_pct"] = comparison_df.apply(
    lambda row: (
        (row["execution_time_delta_ms"] / row["execution_time_ms_before"]) * 100
        if row["execution_time_ms_before"] not in [None, 0]
        else None
    ),
    axis=1
)

comparison_df["docs_examined_improvement_pct"] = comparison_df.apply(
    lambda row: (
        (row["docs_examined_delta"] / row["total_docs_examined_before"]) * 100
        if row["total_docs_examined_before"] not in [None, 0]
        else None
    ),
    axis=1
)

comparison_df[
    [
        "query_code",
        "execution_time_ms_before",
        "execution_time_ms_after",
        "execution_time_improvement_pct",
        "total_docs_examined_before",
        "total_docs_examined_after",
        "docs_examined_improvement_pct",
        "plan_classification_before",
        "plan_classification_after",
        "indexes_used_after"
    ]
]

,query_code,execution_time_ms_before,execution_time_ms_after,execution_time_improvement_pct,total_docs_examined_before,total_docs_examined_after,docs_examined_improvement_pct,plan_classification_before,plan_classification_after,indexes_used_after
0,Q01,1,5,-400.000000,2,2,0.0,IXSCAN - Uso de índice,IXSCAN - Uso de índice,"[idx_catalog_category_price, idx_catalog_categ..."
1,Q02,0,0,NaN,1,1,0.0,IXSCAN - Uso de índice,IXSCAN - Uso de índice,"[idx_catalog_category_price, idx_catalog_categ..."
2,Q03,0,3,NaN,1,1,0.0,IXSCAN - Uso de índice,IXSCAN - Uso de índice,"[idx_catalog_seller_name, idx_seller_summary_s..."
3,Q04,4,20,-400.000000,1000,1000,0.0,COLLSCAN - Revisión completa de colección,IXSCAN - Uso de índice,[idx_text_product_search]
4,Q05,0,6,NaN,1,1,0.0,IXSCAN - Uso de índice,IXSCAN - Uso de índice,"[idx_reviews_product_date, idx_reviews_product..."
5,Q06,42,106,-152.380952,15275,15275,0.0,IXSCAN - Uso de índice,IXSCAN - Uso de índice,"[idx_reviews_low_score_partial, idx_reviews_sc..."
6,Q07,4,3,25.000000,0,0,NaN,IXSCAN - Uso de índice,IXSCAN - Uso de índice,"[idx_catalog_category_price, idx_catalog_categ..."


**5.7 Generar interpretación automática**

Esta celda genera una lectura inicial de los resultados. No reemplaza el análisis humano, pero nos ayuda a redactar el documento final con base en las métricas.

In [ ]:
# Hito 5.7 - Interpretación simple de resultados

def interpret_result(row):
    before_plan = row["plan_classification_before"]
    after_plan = row["plan_classification_after"]
    docs_before = row["total_docs_examined_before"]
    docs_after = row["total_docs_examined_after"]
    time_before = row["execution_time_ms_before"]
    time_after = row["execution_time_ms_after"]

    observations = []

    if "COLLSCAN" in str(before_plan) and "IXSCAN" in str(after_plan):
        observations.append("Cambió de escaneo completo a uso de índice.")

    if "TEXT_MATCH" in str(after_plan):
        observations.append("Usó índice de texto para búsqueda full-text.")

    if docs_before is not None and docs_after is not None and docs_after < docs_before:
        observations.append("Redujo documentos examinados.")

    if time_before is not None and time_after is not None and time_after < time_before:
        observations.append("Redujo tiempo de ejecución.")

    if not observations:
        observations.append("No se observa mejora significativa o la consulta ya estaba optimizada.")

    return " ".join(observations)


comparison_df["technical_interpretation"] = comparison_df.apply(
    interpret_result,
    axis=1
)

comparison_df[
    [
        "query_code",
        "technical_interpretation"
    ]
]

,query_code,technical_interpretation
0,Q01,No se observa mejora significativa o la consul...
1,Q02,No se observa mejora significativa o la consul...
2,Q03,No se observa mejora significativa o la consul...
3,Q04,Cambió de escaneo completo a uso de índice.
4,Q05,No se observa mejora significativa o la consul...
5,Q06,No se observa mejora significativa o la consul...
6,Q07,Redujo tiempo de ejecución.


**5.8 Exportar evidencias del Hito 5**

Esta celda guarda la medición posterior y la comparación antes/después.

In [ ]:
# Hito 5.8 - Exportar evidencias del Hito 5

optimized_df.to_csv("hito5_optimized_explain_summary.csv", index=False)
comparison_df.to_csv("hito5_before_after_comparison.csv", index=False)

print("Archivos generados:")
print("- hito5_optimized_explain_summary.csv")
print("- hito5_before_after_comparison.csv")

Archivos generados:
- hito5_optimized_explain_summary.csv
- hito5_before_after_comparison.csv


# Hito 6 — Optimización de aggregation pipeline
**6.1 Objetivo del pipeline analítico**

El pipeline busca responder una pregunta de negocio:
¿Cuáles son las categorías con mejor desempeño comercial y reputacional, considerando productos, ventas y reseñas?
Se va a utilizar product_catalog como colección principal y  $lookup con product_reviews.

**6.2 Definir pipeline base**

Este pipeline base funciona, pero no está organizado de la mejor forma. Hace primero el $lookup y el $unwind, y solo después filtra por categoría. Eso puede generar más trabajo del necesario, porque MongoDB une documentos antes de reducir el conjunto de productos.

In [ ]:
# Hito 6.2 - Pipeline base para análisis de categoría, ventas y reseñas

pipeline_base = [
    {
        "$lookup": {
            "from": "product_reviews",
            "localField": "product_id",
            "foreignField": "product_id",
            "as": "reviews"
        }
    },
    {
        "$unwind": {
            "path": "$reviews",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$match": {
            category_field: selected_category
        }
    },
    {
        "$group": {
            "_id": f"${category_field}",
            "total_products": {"$addToSet": "$product_id"},
            "avg_product_rating": {"$avg": f"${rating_field}"} if rating_field else {"$avg": 0},
            "avg_review_score": {"$avg": f"$reviews.{review_score_field}"} if review_score_field else {"$avg": 0},
            "total_sales": {"$sum": f"${sales_field}"} if sales_field else {"$sum": 0},
            "review_count": {"$sum": 1}
        }
    },
    {
        "$project": {
            "_id": 0,
            "category": "$_id",
            "total_products": {"$size": "$total_products"},
            "avg_product_rating": {"$round": ["$avg_product_rating", 2]},
            "avg_review_score": {"$round": ["$avg_review_score", 2]},
            "total_sales": 1,
            "review_count": 1
        }
    },
    {
        "$sort": {
            "total_sales": -1
        }
    }
]

pipeline_base

[{'$lookup': {'from': 'product_reviews',
   'localField': 'product_id',
   'foreignField': 'product_id',
   'as': 'reviews'}},
 {'$unwind': {'path': '$reviews', 'preserveNullAndEmptyArrays': True}},
 {'$match': {'category.name_translated': 'agro_industry_and_commerce'}},
 {'$group': {'_id': '$category.name_translated',
   'total_products': {'$addToSet': '$product_id'},
   'avg_product_rating': {'$avg': 0},
   'avg_review_score': {'$avg': '$reviews.score'},
   'total_sales': {'$sum': '$sales_summary.total_orders'},
   'review_count': {'$sum': 1}}},
 {'$project': {'_id': 0,
   'category': '$_id',
   'total_products': {'$size': '$total_products'},
   'avg_product_rating': {'$round': ['$avg_product_rating', 2]},
   'avg_review_score': {'$round': ['$avg_review_score', 2]},
   'total_sales': 1,
   'review_count': 1}},
 {'$sort': {'total_sales': -1}}]

**6.3 Medir pipeline base con explain**

Esta celda mide el pipeline base usando executionStats. Se guardan métricas como tiempo, documentos examinados, llaves examinadas, documentos retornados, índices usados y etapas detectadas.

In [ ]:
# Hito 6.3 - Medición del pipeline base con explain

pipeline_base_explain = db.command(
    "explain",
    {
        "aggregate": "product_catalog",
        "pipeline": pipeline_base,
        "cursor": {},
        "allowDiskUse": True
    },
    verbosity="executionStats"
)

pipeline_base_summary = summarize_explain(
    query_code="P01_BASE",
    collection_name="product_catalog",
    operation="aggregate",
    explain_result=pipeline_base_explain
)

pipeline_base_df = pd.DataFrame([pipeline_base_summary])
pipeline_base_df

,query_code,collection,operation,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,stages_detected,captured_at
0,P01_BASE,product_catalog,aggregate,1,3,3,2,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_DEFAULT]",2026-06-15T20:00:49


**6.4 Definir pipeline optimizado**

Esta versión arma el pipeline de forma dinámica. Es decir, solo incluye rating_field, sales_field y review_score_field si realmente existen. Así se evita que Python genere llaves None dentro del pipeline.

In [ ]:
# Hito 6.4 - Pipeline optimizado con construcción segura de campos

# Validación mínima de campos requeridos
if not category_field:
    raise ValueError("No se encontró category_field. Revisar selección de campos del Hito 2.")

if not selected_category:
    raise ValueError("No se encontró selected_category. Revisar parámetros del Hito 2.")

# Project temprano: solo agregamos campos si existen
early_project = {
    "_id": 0,
    "product_id": 1,
    "name": 1,
    category_field: 1
}

if rating_field:
    early_project[rating_field] = 1

if sales_field:
    early_project[sales_field] = 1

# Group dinámico: solo calcula métricas si los campos existen
group_stage = {
    "_id": f"${category_field}",
    "total_products": {"$addToSet": "$product_id"},
    "review_count": {"$sum": 1}
}

if rating_field:
    group_stage["avg_product_rating"] = {"$avg": f"${rating_field}"}

if review_score_field:
    group_stage["avg_review_score"] = {"$avg": f"$reviews.{review_score_field}"}

if sales_field:
    group_stage["total_sales"] = {"$sum": f"${sales_field}"}
else:
    group_stage["total_sales"] = {"$sum": 0}

# Project final dinámico
final_project = {
    "_id": 0,
    "category": "$_id",
    "total_products": {"$size": "$total_products"},
    "total_sales": 1,
    "review_count": 1
}

if rating_field:
    final_project["avg_product_rating"] = {"$round": ["$avg_product_rating", 2]}

if review_score_field:
    final_project["avg_review_score"] = {"$round": ["$avg_review_score", 2]}

pipeline_optimized = [
    {
        "$match": {
            category_field: selected_category
        }
    },
    {
        "$project": early_project
    },
    {
        "$lookup": {
            "from": "product_reviews",
            "localField": "product_id",
            "foreignField": "product_id",
            "as": "reviews"
        }
    },
    {
        "$unwind": {
            "path": "$reviews",
            "preserveNullAndEmptyArrays": True
        }
    },
    {
        "$group": group_stage
    },
    {
        "$project": final_project
    },
    {
        "$sort": {
            "total_sales": -1
        }
    }
]

pipeline_optimized

[{'$match': {'category.name_translated': 'agro_industry_and_commerce'}},
 {'$project': {'_id': 0,
   'product_id': 1,
   'name': 1,
   'category.name_translated': 1,
   'sales_summary.total_orders': 1}},
 {'$lookup': {'from': 'product_reviews',
   'localField': 'product_id',
   'foreignField': 'product_id',
   'as': 'reviews'}},
 {'$unwind': {'path': '$reviews', 'preserveNullAndEmptyArrays': True}},
 {'$group': {'_id': '$category.name_translated',
   'total_products': {'$addToSet': '$product_id'},
   'review_count': {'$sum': 1},
   'avg_review_score': {'$avg': '$reviews.score'},
   'total_sales': {'$sum': '$sales_summary.total_orders'}}},
 {'$project': {'_id': 0,
   'category': '$_id',
   'total_products': {'$size': '$total_products'},
   'total_sales': 1,
   'review_count': 1,
   'avg_review_score': {'$round': ['$avg_review_score', 2]}}},
 {'$sort': {'total_sales': -1}}]

**6.5 Ejecutar pipeline optimizado**

Esta celda ejecuta el pipeline optimizado para validar que produce resultados correctos. Todavía no compara rendimiento; solo confirma que la consulta analítica funciona.

In [ ]:
# Hito 6.5 - Ejecución funcional del pipeline optimizado

pipeline_optimized_result = list(
    product_catalog.aggregate(
        pipeline_optimized,
        allowDiskUse=True
    )
)

pipeline_optimized_result_df = pd.DataFrame(pipeline_optimized_result)
pipeline_optimized_result_df

,review_count,total_sales,category,total_products,avg_review_score
0,2,2,agro_industry_and_commerce,2,5.0


**6.6 Medir pipeline optimizado con explain**

Esta celda mide el pipeline optimizado. La expectativa es que MongoDB examine menos documentos antes del $lookup y aproveche mejor el índice por categoría si está disponible.

In [ ]:
# Hito 6.6 - Medición del pipeline optimizado con explain

pipeline_optimized_explain = db.command(
    "explain",
    {
        "aggregate": "product_catalog",
        "pipeline": pipeline_optimized,
        "cursor": {},
        "allowDiskUse": True
    },
    verbosity="executionStats"
)

pipeline_optimized_summary = summarize_explain(
    query_code="P01_OPTIMIZED",
    collection_name="product_catalog",
    operation="aggregate",
    explain_result=pipeline_optimized_explain
)

pipeline_optimized_df = pd.DataFrame([pipeline_optimized_summary])
pipeline_optimized_df

,query_code,collection,operation,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,stages_detected,captured_at
0,P01_OPTIMIZED,product_catalog,aggregate,1,3,3,2,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_DEFAULT]",2026-06-15T20:19:48


**6.7 Comparar pipeline base vs pipeline optimizado**

Esta tabla compara el pipeline base contra el optimizado. Para el documento final nos interesa evidenciar si bajó el tiempo, si disminuyeron documentos examinados o si el plan mostró uso de índices.

In [ ]:
# Hito 6.7 - Comparación pipeline base vs optimizado

pipeline_comparison_df = pd.DataFrame([
    {
        "version": "BASE",
        "execution_time_ms": pipeline_base_summary.get("execution_time_ms"),
        "total_docs_examined": pipeline_base_summary.get("total_docs_examined"),
        "total_keys_examined": pipeline_base_summary.get("total_keys_examined"),
        "n_returned": pipeline_base_summary.get("n_returned"),
        "indexes_used": pipeline_base_summary.get("indexes_used"),
        "stages_detected": pipeline_base_summary.get("stages_detected")
    },
    {
        "version": "OPTIMIZED",
        "execution_time_ms": pipeline_optimized_summary.get("execution_time_ms"),
        "total_docs_examined": pipeline_optimized_summary.get("total_docs_examined"),
        "total_keys_examined": pipeline_optimized_summary.get("total_keys_examined"),
        "n_returned": pipeline_optimized_summary.get("n_returned"),
        "indexes_used": pipeline_optimized_summary.get("indexes_used"),
        "stages_detected": pipeline_optimized_summary.get("stages_detected")
    }
])

pipeline_comparison_df

,version,execution_time_ms,total_docs_examined,total_keys_examined,n_returned,indexes_used,stages_detected
0,BASE,1,3,3,2,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_DEFAULT]"
1,OPTIMIZED,1,3,3,2,"[idx_catalog_category_price, idx_catalog_categ...","[FETCH, IXSCAN, PROJECTION_DEFAULT]"


**6.8 Calcular mejora porcentual**

Esta celda calcula el porcentaje de mejora entre el pipeline base y el pipeline optimizado. En colecciones pequeñas puede que el tiempo no mejore mucho, pero la reducción de documentos examinados o el uso de índices sigue siendo una evidencia válida.

In [ ]:
# Hito 6.8 - Cálculo de mejora porcentual del pipeline

base_time = pipeline_base_summary.get("execution_time_ms")
optimized_time = pipeline_optimized_summary.get("execution_time_ms")

base_docs = pipeline_base_summary.get("total_docs_examined")
optimized_docs = pipeline_optimized_summary.get("total_docs_examined")

pipeline_improvement = {
    "execution_time_improvement_pct": (
        ((base_time - optimized_time) / base_time) * 100
        if base_time not in [None, 0] and optimized_time is not None
        else None
    ),
    "docs_examined_improvement_pct": (
        ((base_docs - optimized_docs) / base_docs) * 100
        if base_docs not in [None, 0] and optimized_docs is not None
        else None
    )
}

pipeline_improvement_df = pd.DataFrame([pipeline_improvement])
pipeline_improvement_df

,execution_time_improvement_pct,docs_examined_improvement_pct
0,0.0,0.0


**6.9 Documentar los stages usados**

Esta celda deja documentado el rol de cada stage.

In [ ]:
# Hito 6.9 - Documentación de stages del pipeline optimizado

pipeline_stages_df = pd.DataFrame([
    {
        "stage_order": 1,
        "stage": "$match",
        "purpose": "Filtrar temprano por categoría para reducir documentos procesados."
    },
    {
        "stage_order": 2,
        "stage": "$project",
        "purpose": "Reducir campos antes de hacer lookup y agrupaciones."
    },
    {
        "stage_order": 3,
        "stage": "$lookup",
        "purpose": "Unir productos del catálogo con sus reseñas."
    },
    {
        "stage_order": 4,
        "stage": "$unwind",
        "purpose": "Descomponer el arreglo de reseñas para calcular métricas."
    },
    {
        "stage_order": 5,
        "stage": "$group",
        "purpose": "Agrupar por categoría y calcular métricas agregadas."
    },
    {
        "stage_order": 6,
        "stage": "$project",
        "purpose": "Formatear la salida final del análisis."
    },
    {
        "stage_order": 7,
        "stage": "$sort",
        "purpose": "Ordenar categorías por desempeño comercial."
    }
])

pipeline_stages_df

,stage_order,stage,purpose
0,1,$match,Filtrar temprano por categoría para reducir do...
1,2,$project,Reducir campos antes de hacer lookup y agrupac...
2,3,$lookup,Unir productos del catálogo con sus reseñas.
3,4,$unwind,Descomponer el arreglo de reseñas para calcula...
4,5,$group,Agrupar por categoría y calcular métricas agre...
5,6,$project,Formatear la salida final del análisis.
6,7,$sort,Ordenar categorías por desempeño comercial.


**6.10 Exportar evidencias del Hito 6**

Esta celda exporta las evidencias principales del Hito 6: comparación base vs optimizada, mejora porcentual y documentación de los stages.

In [ ]:
# Hito 6.10 - Exportar evidencias del Hito 6

pipeline_comparison_df.to_csv("hito6_pipeline_comparison.csv", index=False)
pipeline_improvement_df.to_csv("hito6_pipeline_improvement.csv", index=False)
pipeline_stages_df.to_csv("hito6_pipeline_stages.csv", index=False)

print("Archivos generados:")
print("- hito6_pipeline_comparison.csv")
print("- hito6_pipeline_improvement.csv")
print("- hito6_pipeline_stages.csv")

Archivos generados:
- hito6_pipeline_comparison.csv
- hito6_pipeline_improvement.csv
- hito6_pipeline_stages.csv


# Hito 7 — Sharding y replica sets teórico/simulado
**7.1 Alcance del hito**

En esta celda se ve que el hito es de análisis y simulación, solo se documentan decisiones de diseño.

In [ ]:
# Hito 7.1 - Alcance del análisis de sharding y replica sets

hito7_scope = {
    "implementation_type": "Teórico / simulado",
    "real_sharding_enabled": False,
    "real_replica_set_modified": False,
    "target_collections": ["product_catalog", "product_reviews"],
    "purpose": (
        "Analizar distribución de datos, seleccionar shard keys candidatas "
        "y proponer una estrategia de réplica para el módulo MongoDB de Ecommify."
    )
}

hito7_scope

{'implementation_type': 'Teórico / simulado',
 'real_sharding_enabled': False,
 'real_replica_set_modified': False,
 'target_collections': ['product_catalog', 'product_reviews'],
 'purpose': 'Analizar distribución de datos, seleccionar shard keys candidatas y proponer una estrategia de réplica para el módulo MongoDB de Ecommify.'}

**7.2 Analizar distribución por categoría**

Esta celda calcula cuántos productos existen por categoría. Sirve para detectar concentración de datos, categorías dominantes y posibles riesgos si se eligiera category como única shard key.

In [ ]:
# Hito 7.2 - Distribución de productos por categoría

if not category_field:
    raise ValueError("No se encontró category_field. Revisar Hito 2.")

category_distribution_pipeline = [
    {
        "$match": {
            category_field: {
                "$exists": True,
                "$ne": None
            }
        }
    },
    {
        "$group": {
            "_id": f"${category_field}",
            "documents": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "documents": -1
        }
    }
]

category_distribution = list(
    product_catalog.aggregate(category_distribution_pipeline)
)

category_distribution_df = pd.DataFrame(category_distribution)

if not category_distribution_df.empty:
    category_distribution_df = category_distribution_df.rename(
        columns={"_id": "category"}
    )

category_distribution_df.head(15)

,category,documents
0,bed_bath_table,86
1,sports_leisure,83
2,health_beauty,75
3,furniture_decor,72
4,housewares,64
5,auto,57
6,computers_accessories,45
7,toys,45
8,telephony,38
9,watches_gifts,36


**7.3 Calcular concentración: participación máxima e índice HHI**

Esta celda calcula dos señales simples de concentración. max_category_share muestra qué porcentaje representa la categoría más grande. hhi_category_concentration ayuda a ver si la distribución está muy concentrada o más repartida.

In [ ]:
# Hito 7.3 - Métricas de concentración por categoría

total_category_docs = category_distribution_df["documents"].sum()

category_distribution_df["share"] = (
    category_distribution_df["documents"] / total_category_docs
)

max_share = category_distribution_df["share"].max()

hhi = (category_distribution_df["share"] ** 2).sum()

concentration_metrics_df = pd.DataFrame([
    {
        "metric": "total_documents_analyzed",
        "value": total_category_docs
    },
    {
        "metric": "max_category_share",
        "value": round(max_share, 4)
    },
    {
        "metric": "hhi_category_concentration",
        "value": round(hhi, 4)
    }
])

concentration_metrics_df

,metric,value
0,total_documents_analyzed,971.0000
1,max_category_share,0.0886
2,hhi_category_concentration,0.0483


**7.4 Preparar muestra para simulación de shards**

Esta celda extrae product_id y categoría para simular cómo se podrían distribuir los documentos entre shards. No modifica MongoDB; solo construye una tabla local en Colab.

In [ ]:
# Hito 7.4 - Preparación de datos para simulación de distribución en shards

import hashlib

def get_nested_value(document, dotted_field):
    """
    Obtiene un valor desde un documento usando una ruta con puntos.
    Ejemplo: category.name_translated
    """
    current = document

    for part in dotted_field.split("."):
        if isinstance(current, dict):
            current = current.get(part)
        else:
            return None

    return current


def stable_hash(value):
    """
    Genera un hash estable para simular asignación a shards.
    """
    value_as_text = str(value).encode("utf-8")
    return int(hashlib.md5(value_as_text).hexdigest(), 16)


projection = {
    "_id": 0,
    "product_id": 1,
    category_field: 1
}

sample_docs = list(
    product_catalog.find({}, projection)
)

simulation_rows = []

for doc in sample_docs:
    product_id_value = doc.get("product_id")
    category_value = get_nested_value(doc, category_field)

    simulation_rows.append({
        "product_id": product_id_value,
        "category": category_value
    })

sharding_sample_df = pd.DataFrame(simulation_rows)

sharding_sample_df.head()

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares


**7.5 Simular tres estrategias de shard key**

Esta celda compara tres alternativas: distribuir por categoría, por hash de product_id y por combinación categoría-producto. La idea es observar cuál genera una distribución más balanceada.

In [ ]:
# Hito 7.5 - Simulación de distribución con 3 shards

NUM_SHARDS = 3

simulation_df = sharding_sample_df.dropna(
    subset=["product_id", "category"]
).copy()

simulation_df["shard_by_category"] = simulation_df["category"].apply(
    lambda value: f"shard_{stable_hash(value) % NUM_SHARDS}"
)

simulation_df["shard_by_product_hash"] = simulation_df["product_id"].apply(
    lambda value: f"shard_{stable_hash(value) % NUM_SHARDS}"
)

simulation_df["shard_by_category_product"] = simulation_df.apply(
    lambda row: f"shard_{stable_hash(str(row['category']) + '_' + str(row['product_id'])) % NUM_SHARDS}",
    axis=1
)

simulation_df.head()

,product_id,category,shard_by_category,shard_by_product_hash,shard_by_category_product
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,shard_2,shard_1,shard_2
1,3aa071139cb16b67ca9e5dea641aaa2f,art,shard_0,shard_1,shard_2
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,shard_0,shard_0,shard_0
3,cef67bcfe19066a932b7673e239eb23d,baby,shard_2,shard_2,shard_1
4,9dc1a7de274444849c219cff195d0b71,housewares,shard_1,shard_0,shard_1


**7.6 Comparar distribución simulada**

Esta celda resume qué tan balanceada queda cada estrategia. Una shard key buena debería evitar que un solo shard concentre demasiados documentos.

In [ ]:
# Hito 7.6 - Comparación de balance entre estrategias

def summarize_shard_distribution(df, strategy_column):
    distribution = (
        df.groupby(strategy_column)
        .size()
        .reset_index(name="documents")
        .rename(columns={strategy_column: "shard"})
    )

    total_docs = distribution["documents"].sum()
    distribution["share"] = distribution["documents"] / total_docs

    max_share_value = distribution["share"].max()
    min_share_value = distribution["share"].min()
    imbalance_value = max_share_value - min_share_value

    return distribution, {
        "strategy": strategy_column,
        "total_documents": total_docs,
        "max_share": round(max_share_value, 4),
        "min_share": round(min_share_value, 4),
        "imbalance": round(imbalance_value, 4)
    }


distribution_tables = []
strategy_metrics = []

for strategy in [
    "shard_by_category",
    "shard_by_product_hash",
    "shard_by_category_product"
]:
    distribution, metrics = summarize_shard_distribution(simulation_df, strategy)
    distribution["strategy"] = strategy

    distribution_tables.append(distribution)
    strategy_metrics.append(metrics)

shard_distribution_df = pd.concat(distribution_tables, ignore_index=True)
shard_strategy_metrics_df = pd.DataFrame(strategy_metrics)

display(shard_distribution_df)
display(shard_strategy_metrics_df)

,shard,documents,share,strategy
0,shard_0,438,0.451081,shard_by_category
1,shard_1,243,0.250257,shard_by_category
2,shard_2,290,0.298661,shard_by_category
3,shard_0,335,0.345005,shard_by_product_hash
4,shard_1,310,0.319258,shard_by_product_hash
5,shard_2,326,0.335736,shard_by_product_hash
6,shard_0,321,0.330587,shard_by_category_product
7,shard_1,314,0.323378,shard_by_category_product
8,shard_2,336,0.346035,shard_by_category_product


,strategy,total_documents,max_share,min_share,imbalance
0,shard_by_category,971,0.4511,0.2503,0.2008
1,shard_by_product_hash,971,0.3450,0.3193,0.0257
2,shard_by_category_product,971,0.3460,0.3234,0.0227


**7.7 Matriz de decisión de shard keys**

Esta celda documenta la decisión. Para product_catalog, se recomienda una clave compuesta teórica basada en categoría y producto. Para product_reviews, product_id es mas natural porque las reseñas se consultan frecuentemente por producto.

In [ ]:
# Hito 7.7 - Matriz de decisión de shard keys candidatas

shard_key_decision_df = pd.DataFrame([
    {
        "collection": "product_catalog",
        "candidate_shard_key": "category",
        "advantages": "Ayuda a consultas por categoría.",
        "risks": "Puede generar hotspots si pocas categorías concentran muchos productos.",
        "decision": "No seleccionada como única clave."
    },
    {
        "collection": "product_catalog",
        "candidate_shard_key": "hashed product_id",
        "advantages": "Distribuye mejor los documentos.",
        "risks": "Las consultas por categoría podrían requerir búsqueda en varios shards.",
        "decision": "Candidata para balance, pero limitada para navegación por categoría."
    },
    {
        "collection": "product_catalog",
        "candidate_shard_key": "category + hashed product_id",
        "advantages": "Combina localidad por categoría con mejor distribución por producto.",
        "risks": "Mayor complejidad de diseño e índices compatibles.",
        "decision": "Seleccionada como estrategia teórica principal."
    },
    {
        "collection": "product_reviews",
        "candidate_shard_key": "hashed product_id",
        "advantages": "Distribuye reseñas y favorece consultas por producto.",
        "risks": "Agregaciones globales pueden requerir fan-out.",
        "decision": "Seleccionada como estrategia teórica para reseñas."
    }
])

shard_key_decision_df

,collection,candidate_shard_key,advantages,risks,decision
0,product_catalog,category,Ayuda a consultas por categoría.,Puede generar hotspots si pocas categorías con...,No seleccionada como única clave.
1,product_catalog,hashed product_id,Distribuye mejor los documentos.,Las consultas por categoría podrían requerir b...,"Candidata para balance, pero limitada para nav..."
2,product_catalog,category + hashed product_id,Combina localidad por categoría con mejor dist...,Mayor complejidad de diseño e índices compatib...,Seleccionada como estrategia teórica principal.
3,product_reviews,hashed product_id,Distribuye reseñas y favorece consultas por pr...,Agregaciones globales pueden requerir fan-out.,Seleccionada como estrategia teórica para rese...


**7.8 Comandos teóricos de sharding**

Esta celda solo documenta comandos teóricos. No se deben ejecutar en el Colab. Sirven como evidencia de diseño para el documento final.

In [ ]:
# Hito 7.8 - Comandos teóricos de sharding, NO ejecutar en este entorno

theoretical_sharding_commands_df = pd.DataFrame([
    {
        "order": 1,
        "command": 'sh.enableSharding("ecommify_mongodb")',
        "purpose": "Habilitar sharding para la base de datos."
    },
    {
        "order": 2,
        "command": (
            'sh.shardCollection("ecommify_mongodb.product_catalog", '
            '{"category.name_translated": 1, "product_id": "hashed"})'
        ),
        "purpose": "Distribuir catálogo con estrategia compuesta categoría + producto."
    },
    {
        "order": 3,
        "command": (
            'sh.shardCollection("ecommify_mongodb.product_reviews", '
            '{"product_id": "hashed"})'
        ),
        "purpose": "Distribuir reseñas usando product_id como clave estable."
    }
])

theoretical_sharding_commands_df

,order,command,purpose
0,1,"sh.enableSharding(""ecommify_mongodb"")",Habilitar sharding para la base de datos.
1,2,"sh.shardCollection(""ecommify_mongodb.product_c...",Distribuir catálogo con estrategia compuesta c...
2,3,"sh.shardCollection(""ecommify_mongodb.product_r...",Distribuir reseñas usando product_id como clav...


**7.9 Estrategia teórica de replica set**

Esta celda define una arquitectura teórica de replica set con un nodo primario y dos secundarios. Es útil para explicar alta disponibilidad y separación parcial de lecturas.

In [ ]:
# Hito 7.9 - Estrategia teórica de replica set

replica_set_strategy_df = pd.DataFrame([
    {
        "component": "Primary node",
        "role": "Recibe escrituras y operaciones críticas.",
        "ecommify_usage": "Actualización de catálogo, carga de reseñas y cambios controlados."
    },
    {
        "component": "Secondary node 1",
        "role": "Replica datos y puede atender lecturas no críticas.",
        "ecommify_usage": "Consultas analíticas o de catálogo con tolerancia a ligera latencia."
    },
    {
        "component": "Secondary node 2",
        "role": "Aumenta disponibilidad y soporta failover.",
        "ecommify_usage": "Continuidad operativa ante falla del primary."
    }
])

replica_set_strategy_df

,component,role,ecommify_usage
0,Primary node,Recibe escrituras y operaciones críticas.,"Actualización de catálogo, carga de reseñas y ..."
1,Secondary node 1,Replica datos y puede atender lecturas no crít...,Consultas analíticas o de catálogo con toleran...
2,Secondary node 2,Aumenta disponibilidad y soporta failover.,Continuidad operativa ante falla del primary.


**7.10 Read preference, read concern y write concern**

Esta celda conecta la estrategia de réplica con las consultas del proyecto. No todo requiere la misma consistencia: catálogo y analítica toleran más latencia; cargas o datos recientes requieren más cuidado.

In [ ]:
# Hito 7.10 - Estrategia de lectura y escritura

read_write_strategy_df = pd.DataFrame([
    {
        "operation_type": "Lectura de catálogo",
        "example_queries": "Q01, Q02, Q03, Q04",
        "read_preference": "secondaryPreferred",
        "read_concern": "local",
        "write_concern": "No aplica",
        "justification": "Son consultas de lectura donde una pequeña latencia de réplica puede ser aceptable."
    },
    {
        "operation_type": "Lectura de reseñas por producto",
        "example_queries": "Q05",
        "read_preference": "primaryPreferred",
        "read_concern": "majority",
        "write_concern": "No aplica",
        "justification": "Las reseñas recientes pueden requerir mayor consistencia."
    },
    {
        "operation_type": "Análisis de reseñas críticas",
        "example_queries": "Q06",
        "read_preference": "secondaryPreferred",
        "read_concern": "local",
        "write_concern": "No aplica",
        "justification": "Consulta analítica tolerante a ligera consistencia eventual."
    },
    {
        "operation_type": "Carga o actualización documental",
        "example_queries": "Inserción/actualización de product_catalog y product_reviews",
        "read_preference": "No aplica",
        "read_concern": "No aplica",
        "write_concern": "majority",
        "justification": "Se prioriza durabilidad y confirmación de escritura replicada."
    }
])

read_write_strategy_df

,operation_type,example_queries,read_preference,read_concern,write_concern,justification
0,Lectura de catálogo,"Q01, Q02, Q03, Q04",secondaryPreferred,local,No aplica,Son consultas de lectura donde una pequeña lat...
1,Lectura de reseñas por producto,Q05,primaryPreferred,majority,No aplica,Las reseñas recientes pueden requerir mayor co...
2,Análisis de reseñas críticas,Q06,secondaryPreferred,local,No aplica,Consulta analítica tolerante a ligera consiste...
3,Carga o actualización documental,Inserción/actualización de product_catalog y p...,No aplica,No aplica,majority,Se prioriza durabilidad y confirmación de escr...


**7.11 Resumen de decisión del Hito 7**

Esta celda consolida las decisiones principales del hito.

In [ ]:
# Hito 7.11 - Resumen de decisión técnica

hito7_decision_summary_df = pd.DataFrame([
    {
        "topic": "Shard key product_catalog",
        "decision": "category + hashed product_id",
        "reason": "Equilibra consultas por categoría con mejor distribución de productos."
    },
    {
        "topic": "Shard key product_reviews",
        "decision": "hashed product_id",
        "reason": "Las reseñas se consultan principalmente por producto."
    },
    {
        "topic": "Replica set",
        "decision": "1 primary + 2 secondaries",
        "reason": "Permite alta disponibilidad y lecturas no críticas desde réplicas."
    },
    {
        "topic": "Write concern",
        "decision": "majority para cargas y actualizaciones",
        "reason": "Aumenta garantía de durabilidad en el módulo documental."
    },
    {
        "topic": "Read preference",
        "decision": "secondaryPreferred para analítica; primaryPreferred para datos recientes",
        "reason": "Balancea rendimiento y consistencia según el tipo de consulta."
    }
])

hito7_decision_summary_df

,topic,decision,reason
0,Shard key product_catalog,category + hashed product_id,Equilibra consultas por categoría con mejor di...
1,Shard key product_reviews,hashed product_id,Las reseñas se consultan principalmente por pr...
2,Replica set,1 primary + 2 secondaries,Permite alta disponibilidad y lecturas no crít...
3,Write concern,majority para cargas y actualizaciones,Aumenta garantía de durabilidad en el módulo d...
4,Read preference,secondaryPreferred para analítica; primaryPref...,Balancea rendimiento y consistencia según el t...


**7.12 Exportar evidencias del Hito 7**

Esta celda exporta las evidencias de distribución, simulación y decisiones arquitectónicas.

In [ ]:
# Hito 7.12 - Exportar evidencias del Hito 7

category_distribution_df.to_csv("hito7_category_distribution.csv", index=False)
concentration_metrics_df.to_csv("hito7_concentration_metrics.csv", index=False)
shard_distribution_df.to_csv("hito7_shard_distribution_simulation.csv", index=False)
shard_strategy_metrics_df.to_csv("hito7_shard_strategy_metrics.csv", index=False)
shard_key_decision_df.to_csv("hito7_shard_key_decision.csv", index=False)
theoretical_sharding_commands_df.to_csv("hito7_theoretical_sharding_commands.csv", index=False)
replica_set_strategy_df.to_csv("hito7_replica_set_strategy.csv", index=False)
read_write_strategy_df.to_csv("hito7_read_write_strategy.csv", index=False)
hito7_decision_summary_df.to_csv("hito7_decision_summary.csv", index=False)

print("Archivos generados:")
print("- hito7_category_distribution.csv")
print("- hito7_concentration_metrics.csv")
print("- hito7_shard_distribution_simulation.csv")
print("- hito7_shard_strategy_metrics.csv")
print("- hito7_shard_key_decision.csv")
print("- hito7_theoretical_sharding_commands.csv")
print("- hito7_replica_set_strategy.csv")
print("- hito7_read_write_strategy.csv")
print("- hito7_decision_summary.csv")

Archivos generados:
- hito7_category_distribution.csv
- hito7_concentration_metrics.csv
- hito7_shard_distribution_simulation.csv
- hito7_shard_strategy_metrics.csv
- hito7_shard_key_decision.csv
- hito7_theoretical_sharding_commands.csv
- hito7_replica_set_strategy.csv
- hito7_read_write_strategy.csv
- hito7_decision_summary.csv


# Hito 8 — Cierre de evidencias para el documento Word
**8.1 Confirmar estado final de hitos ejecutados**

Esta celda resume el estado de los hitos realizados. Sirve para demostrar que la implementación siguió una ruta ordenada y trazable.

In [ ]:
# Hito 8.1 - Confirmación de hitos ejecutados

hitos_status_df = pd.DataFrame([
    {
        "hito": "Hito 1",
        "nombre": "Conexión y reconocimiento de colecciones",
        "estado": "Completado",
        "evidencia_principal": "Conexión exitosa, conteos e índices iniciales."
    },
    {
        "hito": "Hito 2",
        "nombre": "Definición de consultas críticas",
        "estado": "Completado",
        "evidencia_principal": "Q01-Q07 definidas y validadas funcionalmente."
    },
    {
        "hito": "Hito 3",
        "nombre": "Medición base con explain",
        "estado": "Completado",
        "evidencia_principal": "baseline_df y CSV de línea base."
    },
    {
        "hito": "Hito 4",
        "nombre": "Creación y reutilización de índices",
        "estado": "Completado",
        "evidencia_principal": "Índices creados/reutilizados, inventario final de índices."
    },
    {
        "hito": "Hito 5",
        "nombre": "Comparación antes/después",
        "estado": "Completado",
        "evidencia_principal": "comparison_df con mejoras por consulta."
    },
    {
        "hito": "Hito 6",
        "nombre": "Optimización de aggregation pipeline",
        "estado": "Completado",
        "evidencia_principal": "Pipeline base vs optimizado y tabla de stages."
    },
    {
        "hito": "Hito 7",
        "nombre": "Sharding y replica sets teórico/simulado",
        "estado": "Completado",
        "evidencia_principal": "Simulación de shards, decisión de shard keys y estrategia de réplica."
    },
    {
        "hito": "Hito 8",
        "nombre": "Cierre de evidencias",
        "estado": "En ejecución",
        "evidencia_principal": "Mapa de evidencias para el documento Word."
    }
])

hitos_status_df

,hito,nombre,estado,evidencia_principal
0,Hito 1,Conexión y reconocimiento de colecciones,Completado,"Conexión exitosa, conteos e índices iniciales."
1,Hito 2,Definición de consultas críticas,Completado,Q01-Q07 definidas y validadas funcionalmente.
2,Hito 3,Medición base con explain,Completado,baseline_df y CSV de línea base.
3,Hito 4,Creación y reutilización de índices,Completado,"Índices creados/reutilizados, inventario final..."
4,Hito 5,Comparación antes/después,Completado,comparison_df con mejoras por consulta.
5,Hito 6,Optimización de aggregation pipeline,Completado,Pipeline base vs optimizado y tabla de stages.
6,Hito 7,Sharding y replica sets teórico/simulado,Completado,"Simulación de shards, decisión de shard keys y..."
7,Hito 8,Cierre de evidencias,En ejecución,Mapa de evidencias para el documento Word.


**8.2 Crear mapa de evidencias para el Word**

Cada sección de este mapa tendrá una referencia directa al Colab.

In [ ]:
# Hito 8.2 - Mapa de evidencias para el documento Word

evidence_map_df = pd.DataFrame([
    {
        "seccion_word_sugerida": "1. Resumen ejecutivo",
        "referencia_colab": "Hito 8.1",
        "evidencia": "Estado general de los hitos ejecutados.",
        "uso_en_documento": "Explicar alcance y resultado global de la etapa."
    },
    {
        "seccion_word_sugerida": "2. Entorno de trabajo y colecciones",
        "referencia_colab": "Hito 1",
        "evidencia": "Conexión a MongoDB Atlas, colecciones product_catalog y product_reviews.",
        "uso_en_documento": "Demostrar que el módulo documental ya estaba cargado y disponible."
    },
    {
        "seccion_word_sugerida": "3. Consultas críticas",
        "referencia_colab": "Hito 2.5 - Hito 2.7",
        "evidencia": "Consultas Q01-Q07 y validación funcional.",
        "uso_en_documento": "Justificar los escenarios de optimización."
    },
    {
        "seccion_word_sugerida": "4. Línea base de rendimiento",
        "referencia_colab": "Hito 3.2 - Hito 3.6",
        "evidencia": "baseline_df, baseline_efficiency_df y CSV de línea base.",
        "uso_en_documento": "Presentar métricas antes de optimizar."
    },
    {
        "seccion_word_sugerida": "5. Índices implementados o reutilizados",
        "referencia_colab": "Hito 4.3 - Hito 4.12",
        "evidencia": "created_indexes_df e index_inventory_df.",
        "uso_en_documento": "Explicar índices compuestos, parciales y de texto."
    },
    {
        "seccion_word_sugerida": "6. Comparación antes/después",
        "referencia_colab": "Hito 5.6 - Hito 5.8",
        "evidencia": "comparison_df y archivos hito5.",
        "uso_en_documento": "Mostrar impacto de optimización con métricas."
    },
    {
        "seccion_word_sugerida": "7. Optimización del aggregation pipeline",
        "referencia_colab": "Hito 6.2 - Hito 6.10",
        "evidencia": "pipeline_comparison_df, pipeline_improvement_df y pipeline_stages_df.",
        "uso_en_documento": "Documentar proceso de optimización del pipeline."
    },
    {
        "seccion_word_sugerida": "8. Sharding y replica sets",
        "referencia_colab": "Hito 7.2 - Hito 7.12",
        "evidencia": "Distribución por categoría, simulación de shards y estrategia teórica.",
        "uso_en_documento": "Justificar shard keys, replica set, read/write concerns."
    },
    {
        "seccion_word_sugerida": "9. Conclusiones y limitaciones",
        "referencia_colab": "Hito 8.3 - Hito 8.5",
        "evidencia": "Resumen técnico, limitaciones y archivos exportados.",
        "uso_en_documento": "Cerrar el documento con hallazgos y alcance real."
    }
])

evidence_map_df

,seccion_word_sugerida,referencia_colab,evidencia,uso_en_documento
0,1. Resumen ejecutivo,Hito 8.1,Estado general de los hitos ejecutados.,Explicar alcance y resultado global de la etapa.
1,2. Entorno de trabajo y colecciones,Hito 1,"Conexión a MongoDB Atlas, colecciones product_...",Demostrar que el módulo documental ya estaba c...
2,3. Consultas críticas,Hito 2.5 - Hito 2.7,Consultas Q01-Q07 y validación funcional.,Justificar los escenarios de optimización.
3,4. Línea base de rendimiento,Hito 3.2 - Hito 3.6,"baseline_df, baseline_efficiency_df y CSV de l...",Presentar métricas antes de optimizar.
4,5. Índices implementados o reutilizados,Hito 4.3 - Hito 4.12,created_indexes_df e index_inventory_df.,"Explicar índices compuestos, parciales y de te..."
5,6. Comparación antes/después,Hito 5.6 - Hito 5.8,comparison_df y archivos hito5.,Mostrar impacto de optimización con métricas.
6,7. Optimización del aggregation pipeline,Hito 6.2 - Hito 6.10,"pipeline_comparison_df, pipeline_improvement_d...",Documentar proceso de optimización del pipeline.
7,8. Sharding y replica sets,Hito 7.2 - Hito 7.12,"Distribución por categoría, simulación de shar...","Justificar shard keys, replica set, read/write..."
8,9. Conclusiones y limitaciones,Hito 8.3 - Hito 8.5,"Resumen técnico, limitaciones y archivos expor...",Cerrar el documento con hallazgos y alcance real.


**8.3 Consolidar resumen técnico de resultados**

Esta celda genera un resumen técnico corto de lo realizado.

In [ ]:
# Hito 8.3 - Resumen técnico consolidado

technical_summary_rows = []

# Consultas críticas
if "critical_queries_df" in globals():
    technical_summary_rows.append({
        "tema": "Consultas críticas",
        "resultado": f"Se definieron {len(critical_queries_df)} consultas críticas para catálogo, búsqueda textual, reseñas y analítica."
    })

# Índices
if "created_indexes_df" in globals():
    created_count = len(created_indexes_df[created_indexes_df["status"] == "OK"]) if "status" in created_indexes_df.columns else 0
    reused_count = len(created_indexes_df[created_indexes_df["status"] == "REUSED"]) if "status" in created_indexes_df.columns else 0

    technical_summary_rows.append({
        "tema": "Índices",
        "resultado": f"Se registraron {created_count} índices creados y {reused_count} índices reutilizados para evitar sobreindexación."
    })

# Comparación antes/después
if "comparison_df" in globals():
    technical_summary_rows.append({
        "tema": "Comparación antes/después",
        "resultado": "Se compararon métricas de executionTimeMillis, documentos examinados, llaves examinadas y planes de ejecución."
    })

# Pipeline
if "pipeline_comparison_df" in globals():
    technical_summary_rows.append({
        "tema": "Aggregation pipeline",
        "resultado": "Se construyó un pipeline base y una versión optimizada con $match y $project tempranos."
    })

# Sharding
if "hito7_decision_summary_df" in globals():
    technical_summary_rows.append({
        "tema": "Sharding y replica sets",
        "resultado": "Se realizó simulación de distribución y se propuso estrategia teórica de shard keys, réplica y concerns."
    })

technical_summary_df = pd.DataFrame(technical_summary_rows)
technical_summary_df

,tema,resultado
0,Consultas críticas,Se definieron 7 consultas críticas para catálo...
1,Índices,Se registraron 5 índices creados y 2 índices r...
2,Comparación antes/después,"Se compararon métricas de executionTimeMillis,..."
3,Aggregation pipeline,Se construyó un pipeline base y una versión op...
4,Sharding y replica sets,Se realizó simulación de distribución y se pro...


**8.4 Registrar limitaciones y decisiones de alcance**

Esta celda deja claras las decisiones de alcance.

In [ ]:
# Hito 8.4 - Limitaciones y decisiones de alcance

scope_limitations_df = pd.DataFrame([
    {
        "limitacion_o_decision": "No se habilitó sharding real.",
        "justificacion": "La etapa requiere análisis/simulación y el entorno Atlas gratuito no está orientado a cambios administrativos avanzados de sharding."
    },
    {
        "limitacion_o_decision": "No se modificó el diseño transaccional en PostgreSQL.",
        "justificacion": "La Unidad 5 se enfoca en optimización MongoDB para el módulo analítico/documental de Ecommify."
    },
    {
        "limitacion_o_decision": "No se recargaron datos.",
        "justificacion": "Las colecciones product_catalog y product_reviews ya estaban disponibles desde la continuidad del proyecto."
    },
    {
        "limitacion_o_decision": "Se reutilizaron índices existentes cuando fue posible.",
        "justificacion": "Se evitó sobreindexación y conflictos de índices equivalentes en MongoDB Atlas."
    },
    {
        "limitacion_o_decision": "Las métricas dependen del tamaño actual de las colecciones.",
        "justificacion": "En colecciones pequeñas, la mejora en tiempo puede ser baja; por eso también se analizaron planes, documentos examinados e índices usados."
    }
])

scope_limitations_df

,limitacion_o_decision,justificacion
0,No se habilitó sharding real.,La etapa requiere análisis/simulación y el ent...
1,No se modificó el diseño transaccional en Post...,La Unidad 5 se enfoca en optimización MongoDB ...
2,No se recargaron datos.,Las colecciones product_catalog y product_revi...
3,Se reutilizaron índices existentes cuando fue ...,Se evitó sobreindexación y conflictos de índic...
4,Las métricas dependen del tamaño actual de las...,"En colecciones pequeñas, la mejora en tiempo p..."


**8.5 Consolidar archivos de evidencia generados**

Esta celda revisa qué archivos de evidencia existen en el entorno de Colab.

In [ ]:
# Hito 8.5 - Inventario de archivos de evidencia generados

import os

expected_evidence_files = [
    "hito3_baseline_explain_summary.csv",
    "hito4_created_indexes.csv",
    "hito4_index_inventory.csv",
    "hito5_optimized_explain_summary.csv",
    "hito5_before_after_comparison.csv",
    "hito6_pipeline_comparison.csv",
    "hito6_pipeline_improvement.csv",
    "hito6_pipeline_stages.csv",
    "hito7_category_distribution.csv",
    "hito7_concentration_metrics.csv",
    "hito7_shard_distribution_simulation.csv",
    "hito7_shard_strategy_metrics.csv",
    "hito7_shard_key_decision.csv",
    "hito7_theoretical_sharding_commands.csv",
    "hito7_replica_set_strategy.csv",
    "hito7_read_write_strategy.csv",
    "hito7_decision_summary.csv"
]

evidence_files_df = pd.DataFrame([
    {
        "file_name": file_name,
        "exists": os.path.exists(file_name),
        "size_bytes": os.path.getsize(file_name) if os.path.exists(file_name) else None
    }
    for file_name in expected_evidence_files
])

evidence_files_df

,file_name,exists,size_bytes
0,hito3_baseline_explain_summary.csv,True,1798
1,hito4_created_indexes.csv,True,1458
2,hito4_index_inventory.csv,True,1676
3,hito5_optimized_explain_summary.csv,True,1821
4,hito5_before_after_comparison.csv,True,2689
5,hito6_pipeline_comparison.csv,True,461
6,hito6_pipeline_improvement.csv,True,69
7,hito6_pipeline_stages.csv,True,464
8,hito7_category_distribution.csv,True,2352
9,hito7_concentration_metrics.csv,True,104


**8.6 Exportar tablas finales del Hito 8**

Esta celda exporta las tablas finales del Hito 8.

In [ ]:
# Hito 8.6 - Exportar tablas finales del Hito 8

hitos_status_df.to_csv("hito8_status_hitos.csv", index=False)
evidence_map_df.to_csv("hito8_evidence_map.csv", index=False)
technical_summary_df.to_csv("hito8_technical_summary.csv", index=False)
scope_limitations_df.to_csv("hito8_scope_limitations.csv", index=False)
evidence_files_df.to_csv("hito8_evidence_files_inventory.csv", index=False)

print("Archivos generados:")
print("- hito8_status_hitos.csv")
print("- hito8_evidence_map.csv")
print("- hito8_technical_summary.csv")
print("- hito8_scope_limitations.csv")
print("- hito8_evidence_files_inventory.csv")

Archivos generados:
- hito8_status_hitos.csv
- hito8_evidence_map.csv
- hito8_technical_summary.csv
- hito8_scope_limitations.csv
- hito8_evidence_files_inventory.csv


**8.7 Crear paquete ZIP de evidencias**

Esta celda crea un paquete comprimido con las evidencias exportadas.

In [ ]:
# Hito 8.7 - Crear ZIP con evidencias CSV

import zipfile

all_evidence_files = expected_evidence_files + [
    "hito8_status_hitos.csv",
    "hito8_evidence_map.csv",
    "hito8_technical_summary.csv",
    "hito8_scope_limitations.csv",
    "hito8_evidence_files_inventory.csv"
]

zip_file_name = "ecommify_u5_etapa1_evidencias_colab.zip"

with zipfile.ZipFile(zip_file_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_name in all_evidence_files:
        if os.path.exists(file_name):
            zipf.write(file_name)

print(f"Archivo ZIP generado: {zip_file_name}")

Archivo ZIP generado: ecommify_u5_etapa1_evidencias_colab.zip


**8.8 Checklist de capturas recomendadas**

Esta celda deja un checklist de capturas útiles.

In [ ]:
# Hito 8.8 - Checklist de capturas para el documento Word

screenshots_checklist_df = pd.DataFrame([
    {
        "orden": 1,
        "captura": "MongoDB Atlas mostrando ecommify_mongodb, product_catalog y product_reviews.",
        "uso": "Evidenciar colecciones cargadas."
    },
    {
        "orden": 2,
        "captura": "Resultado de conexión exitosa desde Colab.",
        "uso": "Evidenciar conectividad."
    },
    {
        "orden": 3,
        "captura": "critical_queries_df del Hito 2.",
        "uso": "Mostrar consultas críticas."
    },
    {
        "orden": 4,
        "captura": "baseline_efficiency_df o baseline_df del Hito 3.",
        "uso": "Mostrar línea base."
    },
    {
        "orden": 5,
        "captura": "created_indexes_df e index_inventory_df del Hito 4.",
        "uso": "Mostrar índices creados y reutilizados."
    },
    {
        "orden": 6,
        "captura": "comparison_df del Hito 5.",
        "uso": "Mostrar comparación antes/después."
    },
    {
        "orden": 7,
        "captura": "pipeline_comparison_df y pipeline_stages_df del Hito 6.",
        "uso": "Mostrar optimización del pipeline."
    },
    {
        "orden": 8,
        "captura": "shard_strategy_metrics_df y hito7_decision_summary_df del Hito 7.",
        "uso": "Mostrar simulación de sharding y decisiones."
    },
    {
        "orden": 9,
        "captura": "evidence_map_df del Hito 8.",
        "uso": "Mostrar trazabilidad Colab-Word."
    }
])

screenshots_checklist_df

,orden,captura,uso
0,1,"MongoDB Atlas mostrando ecommify_mongodb, prod...",Evidenciar colecciones cargadas.
1,2,Resultado de conexión exitosa desde Colab.,Evidenciar conectividad.
2,3,critical_queries_df del Hito 2.,Mostrar consultas críticas.
3,4,baseline_efficiency_df o baseline_df del Hito 3.,Mostrar línea base.
4,5,created_indexes_df e index_inventory_df del Hi...,Mostrar índices creados y reutilizados.
5,6,comparison_df del Hito 5.,Mostrar comparación antes/después.
6,7,pipeline_comparison_df y pipeline_stages_df de...,Mostrar optimización del pipeline.
7,8,shard_strategy_metrics_df y hito7_decision_sum...,Mostrar simulación de sharding y decisiones.
8,9,evidence_map_df del Hito 8.,Mostrar trazabilidad Colab-Word.


**8.9 Exportar checklist de capturas**

Esta celda exporta el checklist de capturas.

In [ ]:
# Hito 8.9 - Exportar checklist de capturas

screenshots_checklist_df.to_csv("hito8_screenshots_checklist.csv", index=False)

print("Archivo generado:")
print("- hito8_screenshots_checklist.csv")

Archivo generado:
- hito8_screenshots_checklist.csv


In [ ]:
import os

print(f"El archivo se encuentra en: {os.getcwd()}")
print("Archivos en el directorio actual:")
print(os.listdir('.'))

You can download the file `hito8_screenshots_checklist.csv` directly from the Colab file system. On the left sidebar, click on the folder icon to open the file browser. You should see `hito8_screenshots_checklist.csv` listed in the `/content/` directory. You can right-click on the file and select 'Download' to save it to your local machine.

In [ ]:
from google.colab import files
files.download('hito8_screenshots_checklist.csv')